# 09 - Sentimiento por mención de plato: enfoque híbrido v1

## Objetivo

Este notebook construye la primera versión del módulo de sentimiento por mención de plato para Hidden Gems.

Hasta ahora el flujo ha conseguido:

```text
review gastronómica
→ detección de menciones de platos
→ normalización de platos
```

Ahora queremos avanzar a:

mención de plato normalizada
→ sentimiento asociado a esa mención

## Ejemplo:

```
The service was terrible, but the pasta was delicious.
```
La review completa puede ser negativa o neutral, pero el sentimiento del plato pasta es positivo.

## Enfoque v1

Esta primera versión no entrena todavía un modelo específico. Usa un enfoque híbrido y explicable basado en:

contexto cercano a la mención;
frase/sentence donde aparece el plato;
léxico positivo y negativo;
negaciones;
conectores de contraste;
rating/sentimiento general de la review como señal secundaria;
confianza del modelo NER;
flags de ambigüedad.
Salidas esperadas
`dish_mentions_with_sentiment_v1.jsonl`
`dish_mention_sentiment_summary_v1.json`
`dish_mention_sentiment_review_sample_v1.jsonl`
`dish_mention_sentiment_ambiguous_sample_v1.jsonl`

Estos artefactos servirán para:

alimentar una primera agregación de señales;
preparar un ranking inicial;
generar etiquetas débiles para entrenar más adelante un modelo específico de aspect-based sentiment.

In [23]:
# ============================================================
# 01. Imports y configuración base
# ============================================================

from pathlib import Path
import json
import random
import re
import os
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 350)

print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


In [24]:
# ============================================================
# 02. Detectar entorno y preparar carpetas
# ============================================================

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IS_KAGGLE:
    ENV_NAME = "kaggle"
elif IS_COLAB:
    ENV_NAME = "colab"
else:
    ENV_NAME = "local"

print("Entorno detectado:", ENV_NAME)

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "hidden_gems_ai"

elif IS_COLAB:
    from google.colab import drive

    try:
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")

    except Exception as e:
        print("No se ha podido montar Drive. Se usará almacenamiento temporal.")
        print("Error:", e)

        PROJECT_DIR = Path("/content/hidden_gems_ai")

else:
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs" / "dish_mention_sentiment_hybrid"
MENTIONS_DIR = OUTPUT_DIR / "mentions"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
SAMPLES_DIR = OUTPUT_DIR / "samples"

for path in [
    PROJECT_DIR,
    OUTPUT_DIR,
    MENTIONS_DIR,
    ARTIFACTS_DIR,
    SAMPLES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Entorno detectado: kaggle
PROJECT_DIR: /kaggle/working/hidden_gems_ai
OUTPUT_DIR: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sentiment_hybrid


In [25]:
# ============================================================
# 02. Detectar entorno y preparar carpetas
# ============================================================

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IS_KAGGLE:
    ENV_NAME = "kaggle"
elif IS_COLAB:
    ENV_NAME = "colab"
else:
    ENV_NAME = "local"

print("Entorno detectado:", ENV_NAME)

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "hidden_gems_ai"

elif IS_COLAB:
    from google.colab import drive

    try:
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")

    except Exception as e:
        print("No se ha podido montar Drive. Se usará almacenamiento temporal.")
        print("Error:", e)

        PROJECT_DIR = Path("/content/hidden_gems_ai")

else:
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs" / "dish_mention_sentiment_hybrid"
MENTIONS_DIR = OUTPUT_DIR / "mentions"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
SAMPLES_DIR = OUTPUT_DIR / "samples"

for path in [
    PROJECT_DIR,
    OUTPUT_DIR,
    MENTIONS_DIR,
    ARTIFACTS_DIR,
    SAMPLES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Entorno detectado: kaggle
PROJECT_DIR: /kaggle/working/hidden_gems_ai
OUTPUT_DIR: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sentiment_hybrid


In [26]:
# ============================================================
# 03. Diagnóstico de archivos disponibles
# ============================================================

if IS_KAGGLE:
    print("Archivos disponibles en /kaggle/input:")

    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("No estás en Kaggle. Se buscarán archivos en el proyecto local.")

Archivos disponibles en /kaggle/input:
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_labels.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_high_precision_with_negatives.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_normalization_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_manual_review_sample_v2_150.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/transformer_sentiment_metrics.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidates_v2.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_inference_summary_full.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_exploration_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_translation_gold_seed_300_es_ready_v1.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidd

In [27]:
# ============================================================
# 04. Localizar archivo de menciones normalizadas v2
# ============================================================

MENTIONS_NORMALIZED_FILENAME = "dish_mentions_normalized_v2.jsonl"
NORMALIZATION_SUMMARY_FILENAME = "dish_normalization_summary_v2.json"

def find_file(filename: str) -> Path:
    candidate_paths = []

    if IS_KAGGLE:
        candidate_paths.extend(list(Path("/kaggle/input").rglob(filename)))

    candidate_paths.extend(list(PROJECT_DIR.rglob(filename)))
    candidate_paths.extend(list(Path.cwd().rglob(filename)))

    candidate_paths = list(dict.fromkeys(candidate_paths))

    if not candidate_paths:
        raise FileNotFoundError(
            f"No se ha encontrado el archivo requerido: {filename}\n"
            "En Kaggle, asegúrate de haberlo añadido desde Add Data."
        )

    return candidate_paths[0]


MENTIONS_NORMALIZED_PATH = find_file(MENTIONS_NORMALIZED_FILENAME)

try:
    NORMALIZATION_SUMMARY_PATH = find_file(NORMALIZATION_SUMMARY_FILENAME)
except FileNotFoundError:
    NORMALIZATION_SUMMARY_PATH = None

print("MENTIONS_NORMALIZED_PATH:", MENTIONS_NORMALIZED_PATH)
print("NORMALIZATION_SUMMARY_PATH:", NORMALIZATION_SUMMARY_PATH)

MENTIONS_NORMALIZED_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_normalized_v2.jsonl
NORMALIZATION_SUMMARY_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_normalization_summary_v2.json


In [28]:
# ============================================================
# 05. Funciones auxiliares de carga y guardado
# ============================================================

def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    invalid_json_count = 0

    with path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Leyendo {path.name}"):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                invalid_json_count += 1

    print(f"Archivo: {path.name}")
    print(f"Registros cargados: {len(records):,}")
    print(f"Líneas JSON inválidas: {invalid_json_count:,}")

    return pd.DataFrame(records)


def make_json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, set):
        return sorted(list(value))

    return value


def save_jsonl(df_to_save: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in df_to_save.to_dict(orient="records"):
            clean_record = {
                str(key): make_json_safe(value)
                for key, value in record.items()
            }

            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

    print(f"Guardado: {path} ({len(df_to_save):,} registros)")

In [29]:
# ============================================================
# 06. Cargar menciones normalizadas v2
# ============================================================

mentions_raw_df = load_jsonl(MENTIONS_NORMALIZED_PATH)

print("Shape:", mentions_raw_df.shape)

print("\nColumnas:")
print(mentions_raw_df.columns.tolist())

display(mentions_raw_df.head(5))

if NORMALIZATION_SUMMARY_PATH is not None:
    with NORMALIZATION_SUMMARY_PATH.open("r", encoding="utf-8") as f:
        normalization_summary = json.load(f)

    print("\nResumen de normalización v2:")
    print(json.dumps(normalization_summary, indent=2, ensure_ascii=False)[:3000])

Leyendo dish_mentions_normalized_v2.jsonl: 0it [00:00, ?it/s]

Archivo: dish_mentions_normalized_v2.jsonl
Registros cargados: 95,061
Líneas JSON inválidas: 0
Shape: (95061, 48)

Columnas:
['mention_id', 'corpus_document_id', 'review_id', 'business_id', 'business_name', 'city', 'state', 'split', 'rating_value', 'sentiment_label', 'language', 'source_system_code', 'source_dataset', 'dish_mention_text', 'dish_mention_normalized', 'start_char', 'end_char', 'start_token', 'end_token', 'token_count', 'confidence_mean', 'confidence_min', 'confidence_max', 'tokens', 'review_text', 'was_review_truncated', 'model_name', 'model_checkpoint', 'inference_run_name', 'human_review_status', 'normalization_status', 'mention_token_count', 'mention_char_length', 'mention_clean', 'mention_canonical_rule', 'normalization_flags', 'is_noise_candidate', 'normalization_confidence_hint', 'canonical_dish_name', 'dish_id', 'normalization_method', 'canonical_dish_name_v1', 'canonical_dish_name_v2', 'normalization_flags_v2', 'is_noise_candidate_v2', 'normalization_status_v2', '

,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status,mention_token_count,mention_char_length,mention_clean,mention_canonical_rule,normalization_flags,is_noise_candidate,normalization_confidence_hint,canonical_dish_name,dish_id,normalization_method,canonical_dish_name_v1,canonical_dish_name_v2,normalization_flags_v2,is_noise_candidate_v2,normalization_status_v2,normalization_method_v2,dish_id_v2
0,dish_mention_c6a233f28ab0ec50a4bf,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,tamale,tamale,88,94,18,19,1,0.997417,0.997417,0.997417,[tamale],"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with no expectations. Next to the Clarion Hotel.",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,1,6,tamale,tamale,[],False,0.9974,tamale,dish_seed_000063,rule_based_v1,tamale,tamale,[],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000060
1,dish_mention_370e33f760028e4e87a3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,lamb curry,lamb curry,54,64,12,14,2,0.996373,0.993416,0.999331,"[lamb, curry]","Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,2,10,lamb curry,lamb curry,[],False,0.9964,lamb curry,dish_seed_000208,rule_based_v1,lamb curry,lamb curry,[],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000205
2,dish_mention_2c51d464763786d9fef2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,korma,korma,69,74,15,16,1,0.997466,0.997466,0.997466,[korma],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,1,5,korma,korma,[],False,0.9975,korma,dish_seed_000078,rule_based_v1,korma,korma,[],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000074
3,dish_mention_6cfa246893992204e58a,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,naan,naan,103,107,22,23,1,0.998627,0.998627,0.998627,[naan],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_review


Resumen de normalización v2:
{
  "notebook": "08_dish_normalization_and_catalog_builder",
  "version": "v2_rule_based_overrides",
  "source_mentions_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_model_v1_full.jsonl",
  "input": {
    "total_mentions": 95061,
    "unique_reviews": 42471,
    "unique_businesses": 4091,
    "unique_surface_forms": 10258,
    "confidence": {
      "min": 0.35621654987335205,
      "mean": 0.975224027584393,
      "median": 0.9990728795528412,
      "max": 0.9997578263282776
    },
    "split_counts": {
      "train": 76050,
      "validation": 9564,
      "test": 9447
    },
    "sentiment_counts": {
      "positive": 64727,
      "negative": 16411,
      "neutral": 13923
    }
  },
  "v1": {
    "normalized_mentions": 94303,
    "pending_or_excluded_mentions": 758,
    "surface_forms": 10258,
    "canonical_dishes_seed": 9401,
    "aliases_seed": 9692
  },
  "v2": {
    "normalized_mentions": 94932,
    "pending_or_

In [30]:
# ============================================================
# 07. Validación de columnas necesarias
# ============================================================

required_cols = [
    "mention_id",
    "review_id",
    "business_id",
    "dish_mention_text",
    "dish_mention_normalized",
    "canonical_dish_name_v2",
    "dish_id_v2",
    "normalization_status_v2",
    "confidence_mean",
    "rating_value",
    "sentiment_label",
    "split",
    "review_text",
    "start_char",
    "end_char",
]

missing_cols = [
    col for col in required_cols
    if col not in mentions_raw_df.columns
]

if missing_cols:
    raise ValueError(f"Faltan columnas obligatorias: {missing_cols}")

print("Columnas obligatorias presentes.")

print("\nNulos principales:")
display(
    mentions_raw_df[required_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nDuplicados mention_id:")
print(mentions_raw_df["mention_id"].duplicated().sum())

Columnas obligatorias presentes.

Nulos principales:


dish_id_v2                 129
review_id                    0
mention_id                   0
business_id                  0
dish_mention_text            0
dish_mention_normalized      0
canonical_dish_name_v2       0
normalization_status_v2      0
confidence_mean              0
rating_value                 0
sentiment_label              0
split                        0
review_text                  0
start_char                   0
end_char                     0
dtype: int64


Duplicados mention_id:
0


In [31]:
# ============================================================
# 08. Preparar dataframe base
# ============================================================

mentions_df = mentions_raw_df.copy()

mentions_df["mention_id"] = mentions_df["mention_id"].astype(str)
mentions_df["review_id"] = mentions_df["review_id"].astype(str)
mentions_df["business_id"] = mentions_df["business_id"].astype(str)

mentions_df["dish_mention_text"] = (
    mentions_df["dish_mention_text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

mentions_df["dish_mention_normalized"] = (
    mentions_df["dish_mention_normalized"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
    .str.strip()
)

mentions_df["canonical_dish_name_v2"] = (
    mentions_df["canonical_dish_name_v2"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
    .str.strip()
)

mentions_df["dish_id_v2"] = mentions_df["dish_id_v2"].astype(str)

mentions_df["normalization_status_v2"] = (
    mentions_df["normalization_status_v2"]
    .astype(str)
    .str.lower()
    .str.strip()
)

mentions_df["confidence_mean"] = pd.to_numeric(
    mentions_df["confidence_mean"],
    errors="coerce"
)

mentions_df["rating_value"] = pd.to_numeric(
    mentions_df["rating_value"],
    errors="coerce"
)

mentions_df["sentiment_label"] = (
    mentions_df["sentiment_label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

mentions_df["split"] = (
    mentions_df["split"]
    .astype(str)
    .str.lower()
    .str.strip()
)

mentions_df["review_text"] = (
    mentions_df["review_text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

mentions_df["start_char"] = pd.to_numeric(
    mentions_df["start_char"],
    errors="coerce"
).astype("Int64")

mentions_df["end_char"] = pd.to_numeric(
    mentions_df["end_char"],
    errors="coerce"
).astype("Int64")

before_count = len(mentions_df)

# Nos quedamos solo con menciones normalizadas v2 válidas.
mentions_df = mentions_df[
    (mentions_df["normalization_status_v2"] == "normalized_rule_based_v2")
    & mentions_df["dish_id_v2"].notna()
    & (mentions_df["dish_id_v2"].astype(str).str.len() > 0)
    & mentions_df["review_text"].notna()
    & (mentions_df["review_text"].str.len() > 0)
    & mentions_df["start_char"].notna()
    & mentions_df["end_char"].notna()
].copy()

after_count = len(mentions_df)

print("Menciones antes:", before_count)
print("Menciones válidas para sentimiento:", after_count)
print("Descartadas:", before_count - after_count)

display(mentions_df.head(5))

Menciones antes: 95061
Menciones válidas para sentimiento: 94932
Descartadas: 129


,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status,mention_token_count,mention_char_length,mention_clean,mention_canonical_rule,normalization_flags,is_noise_candidate,normalization_confidence_hint,canonical_dish_name,dish_id,normalization_method,canonical_dish_name_v1,canonical_dish_name_v2,normalization_flags_v2,is_noise_candidate_v2,normalization_status_v2,normalization_method_v2,dish_id_v2
0,dish_mention_c6a233f28ab0ec50a4bf,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,tamale,tamale,88,94,18,19,1,0.997417,0.997417,0.997417,[tamale],"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with no expectations. Next to the Clarion Hotel.",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,1,6,tamale,tamale,[],False,0.9974,tamale,dish_seed_000063,rule_based_v1,tamale,tamale,[],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000060
1,dish_mention_370e33f760028e4e87a3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,lamb curry,lamb curry,54,64,12,14,2,0.996373,0.993416,0.999331,"[lamb, curry]","Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,2,10,lamb curry,lamb curry,[],False,0.9964,lamb curry,dish_seed_000208,rule_based_v1,lamb curry,lamb curry,[],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000205
2,dish_mention_2c51d464763786d9fef2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,korma,korma,69,74,15,16,1,0.997466,0.997466,0.997466,[korma],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,normalized_rule_based_v1,1,5,korma,korma,[],False,0.9975,korma,dish_seed_000078,rule_based_v1,korma,korma,[],False,normalized_rule_based_v2,rule_based_v2_overrides,dish_seed_v2_000074
3,dish_mention_6cfa246893992204e58a,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,naan,naan,103,107,22,23,1,0.998627,0.998627,0.998627,[naan],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_review

In [32]:
# ============================================================
# 09. QA inicial de menciones para sentimiento
# ============================================================

qa_initial = {
    "total_mentions_for_sentiment": int(len(mentions_df)),
    "unique_reviews": int(mentions_df["review_id"].nunique()),
    "unique_businesses": int(mentions_df["business_id"].nunique()),
    "unique_dishes": int(mentions_df["dish_id_v2"].nunique()),
    "unique_canonical_dish_names": int(mentions_df["canonical_dish_name_v2"].nunique()),
    "confidence": {
        "min": float(mentions_df["confidence_mean"].min()),
        "mean": float(mentions_df["confidence_mean"].mean()),
        "median": float(mentions_df["confidence_mean"].median()),
        "max": float(mentions_df["confidence_mean"].max()),
    },
    "rating_counts": {
        str(k): int(v)
        for k, v in mentions_df["rating_value"].value_counts().sort_index().to_dict().items()
    },
    "review_sentiment_counts": {
        str(k): int(v)
        for k, v in mentions_df["sentiment_label"].value_counts().to_dict().items()
    },
    "split_counts": {
        str(k): int(v)
        for k, v in mentions_df["split"].value_counts().to_dict().items()
    },
    "top_dishes": {
        str(k): int(v)
        for k, v in mentions_df["canonical_dish_name_v2"].value_counts().head(50).to_dict().items()
    }
}

print(json.dumps(qa_initial, indent=2, ensure_ascii=False)[:5000])

print("\nTop platos:")
display(
    mentions_df["canonical_dish_name_v2"]
    .value_counts()
    .head(50)
    .reset_index()
    .rename(columns={
        "index": "canonical_dish_name_v2",
        "canonical_dish_name_v2": "count"
    })
)

print("\nDistribución de rating:")
display(mentions_df["rating_value"].value_counts().sort_index())

print("\nDistribución de sentimiento general de review:")
display(mentions_df["sentiment_label"].value_counts())

{
  "total_mentions_for_sentiment": 94932,
  "unique_reviews": 42461,
  "unique_businesses": 4088,
  "unique_dishes": 9937,
  "unique_canonical_dish_names": 9937,
  "confidence": {
    "min": 0.35621654987335205,
    "mean": 0.97570959051333,
    "median": 0.9990743398666382,
    "max": 0.9997578263282776
  },
  "rating_counts": {
    "1.0": 7363,
    "2.0": 9032,
    "3.0": 13904,
    "4.0": 29471,
    "5.0": 35162
  },
  "review_sentiment_counts": {
    "positive": 64633,
    "negative": 16395,
    "neutral": 13904
  },
  "split_counts": {
    "train": 75951,
    "validation": 9546,
    "test": 9435
  },
  "top_dishes": {
    "pizza": 8751,
    "burger": 8095,
    "fries": 5375,
    "sushi": 4470,
    "tacos": 4436,
    "shrimp": 4421,
    "steak": 2794,
    "ice cream": 2411,
    "wings": 2215,
    "donut": 2175,
    "oysters": 2129,
    "sandwich": 2127,
    "pancake": 2035,
    "crab": 1535,
    "burrito": 1405,
    "pho": 1225,
    "salmon": 1137,
    "ribs": 1095,
    "fried chi

,count,count
0,pizza,8751
1,burger,8095
2,fries,5375
3,sushi,4470
4,tacos,4436
5,shrimp,4421
6,steak,2794
7,ice cream,2411
8,wings,2215
9,donut,2175



Distribución de rating:


rating_value
1.0     7363
2.0     9032
3.0    13904
4.0    29471
5.0    35162
Name: count, dtype: int64


Distribución de sentimiento general de review:


sentiment_label
positive    64633
negative    16395
neutral     13904
Name: count, dtype: int64

## 1. Extracción de contexto local

El sentimiento de una mención de plato no debe depender únicamente del rating general de la review.

Por eso, para cada mención se extraerán varios contextos:

1. **sentence_context**: frase aproximada donde aparece la mención.
2. **window_context**: ventana de caracteres alrededor de la mención.
3. **left_context**: texto inmediatamente anterior.
4. **right_context**: texto inmediatamente posterior.

Estos contextos permitirán detectar expresiones como:

```text
pasta was delicious
burger was dry
fries were cold
sushi was amazing
```

In [33]:
# ============================================================
# 10. Funciones de extracción de contexto
# ============================================================

SENTENCE_BOUNDARY_PATTERN = re.compile(r"[.!?]+")

def safe_int(value, default=None):
    try:
        if pd.isna(value):
            return default
        return int(value)
    except Exception:
        return default


def normalize_space(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


def extract_sentence_context(text: str, start_char: int, end_char: int) -> str:
    """
    Extrae una frase aproximada alrededor de la mención usando signos . ! ?
    """

    if not isinstance(text, str):
        return ""

    text_len = len(text)

    start_char = max(0, min(start_char, text_len))
    end_char = max(0, min(end_char, text_len))

    left_text = text[:start_char]
    right_text = text[end_char:]

    # Buscar último límite de frase antes de la mención
    left_boundaries = list(SENTENCE_BOUNDARY_PATTERN.finditer(left_text))
    if left_boundaries:
        sentence_start = left_boundaries[-1].end()
    else:
        sentence_start = 0

    # Buscar primer límite de frase después de la mención
    right_boundary = SENTENCE_BOUNDARY_PATTERN.search(right_text)
    if right_boundary:
        sentence_end = end_char + right_boundary.end()
    else:
        sentence_end = text_len

    return normalize_space(text[sentence_start:sentence_end])


def extract_window_context(text: str, start_char: int, end_char: int, window_chars: int = 120) -> dict:
    """
    Extrae ventana izquierda/derecha alrededor de la mención.
    """

    if not isinstance(text, str):
        return {
            "left_context": "",
            "mention_context_text": "",
            "right_context": "",
            "window_context": "",
        }

    text_len = len(text)

    start_char = max(0, min(start_char, text_len))
    end_char = max(0, min(end_char, text_len))

    left_start = max(0, start_char - window_chars)
    right_end = min(text_len, end_char + window_chars)

    left_context = normalize_space(text[left_start:start_char])
    mention_text = normalize_space(text[start_char:end_char])
    right_context = normalize_space(text[end_char:right_end])

    window_context = normalize_space(text[left_start:right_end])

    return {
        "left_context": left_context,
        "mention_context_text": mention_text,
        "right_context": right_context,
        "window_context": window_context,
    }


def extract_contexts_for_row(row, window_chars: int = 120) -> dict:
    text = row["review_text"]
    start_char = safe_int(row["start_char"], 0)
    end_char = safe_int(row["end_char"], start_char)

    sentence_context = extract_sentence_context(
        text=text,
        start_char=start_char,
        end_char=end_char,
    )

    window_contexts = extract_window_context(
        text=text,
        start_char=start_char,
        end_char=end_char,
        window_chars=window_chars,
    )

    return {
        "sentence_context": sentence_context,
        **window_contexts,
    }

In [34]:
# ============================================================
# 11. Aplicar extracción de contexto
# ============================================================

context_records = []

for row in tqdm(
    mentions_df.to_dict(orient="records"),
    desc="Extrayendo contextos"
):
    context_records.append(extract_contexts_for_row(row, window_chars=120))

context_df = pd.DataFrame(context_records)

mentions_context_df = pd.concat(
    [
        mentions_df.reset_index(drop=True),
        context_df.reset_index(drop=True)
    ],
    axis=1
)

mentions_context_df["sentence_context_length"] = mentions_context_df["sentence_context"].str.len()
mentions_context_df["window_context_length"] = mentions_context_df["window_context"].str.len()

print("Shape con contextos:", mentions_context_df.shape)

display(
    mentions_context_df[
        [
            "dish_mention_text",
            "canonical_dish_name_v2",
            "rating_value",
            "sentiment_label",
            "sentence_context",
            "window_context",
        ]
    ].head(10)
)

print("\nLongitud sentence_context:")
display(mentions_context_df["sentence_context_length"].describe())

print("\nLongitud window_context:")
display(mentions_context_df["window_context_length"].describe())

Extrayendo contextos:   0%|          | 0/94932 [00:00<?, ?it/s]

Shape con contextos: (94932, 55)


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,sentence_context,window_context
0,tamale,tamale,3.0,neutral,"Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served a"
1,lamb curry,lamb curry,5.0,positive,Our favorite is the lamb curry and korma.,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...g"
2,korma,korma,5.0,positive,Our favorite is the lamb curry and korma.,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and t"
3,naan,naan,5.0,positive,With 10 different kinds of naan!!!,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad"
4,sandwiches,sandwich,4.0,positive,"Really like that sandwiches come w salad, esp after eating too many curds!","p area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too much cheese which I"
5,wings,wings,5.0,positive,Amazingly amazing wings and homemade bleu cheese.,"Amazingly amazing wings and homemade bleu cheese. Had the ribeye: tender, perfectly prepared, delicious. Nice selection of craft beers. Would D"
6,sushi,sushi,3.0,neutral,Our waitress brought our separate sushi orders on one plate so we couldn't really tell who's was who's and forgot several items on an order.,Had a party of 6 here for hibachi. Our waitress brought our separate sushi orders on one plate so we couldn't really tell who's was who's and forgot several items on an order. I understand makin
7,gnocchi with marinara,gnocchi with marinara,4.0,positive,"Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is just that good","Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is ju"
8,baked eggplant appetizer,baked eggplant appetizer,4.0,positive,"Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is just that good","Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is just that good"
9,Sonoran Dog,sonoran dog,4.0,positive,The bun makes the Sonoran Dog.,"The bun makes the Sonoran Dog. It's like a snuggie for the pup. A first, it seems ridiculous and almost like it's going to be too much, exactly like"



Longitud sentence_context:


count    94932.000000
mean        87.554386
std         55.778207
min          3.000000
25%         52.000000
50%         76.000000
75%        109.000000
max       1034.000000
Name: sentence_context_length, dtype: float64


Longitud window_context:


count    94932.000000
mean       218.587431
std         43.093975
min         80.000000
25%        190.000000
50%        244.000000
75%        247.000000
max        302.000000
Name: window_context_length, dtype: float64

## 2. Léxico de sentimiento local

A continuación se define un léxico inicial de términos positivos y negativos frecuentes en reseñas gastronómicas.

Este léxico no pretende cubrir todo el lenguaje, pero sí capturar señales fuertes como:

- `delicious`, `amazing`, `excellent`, `fresh`, `crispy`, `tender`
- `cold`, `dry`, `bland`, `overcooked`, `soggy`, `burnt`, `disappointing`

También se incluyen negaciones y conectores de contraste.

In [35]:
# ============================================================
# 12. Léxicos de sentimiento local
# ============================================================

POSITIVE_LEXICON = {
    "amazing",
    "awesome",
    "excellent",
    "fantastic",
    "perfect",
    "perfectly",
    "delicious",
    "tasty",
    "yummy",
    "great",
    "good",
    "best",
    "favorite",
    "fresh",
    "crispy",
    "crisp",
    "tender",
    "juicy",
    "flavorful",
    "flavourful",
    "rich",
    "creamy",
    "smooth",
    "savory",
    "savoury",
    "well-seasoned",
    "seasoned",
    "hot",
    "warm",
    "melted",
    "melts",
    "love",
    "loved",
    "enjoy",
    "enjoyed",
    "recommend",
    "recommended",
    "worth",
    "solid",
    "wonderful",
    "outstanding",
    "phenomenal",
    "incredible",
    "impressive",
}

NEGATIVE_LEXICON = {
    "bad",
    "awful",
    "terrible",
    "horrible",
    "poor",
    "worst",
    "disappointing",
    "disappointed",
    "bland",
    "boring",
    "flavorless",
    "flavourless",
    "tasteless",
    "cold",
    "dry",
    "soggy",
    "greasy",
    "oily",
    "burnt",
    "overcooked",
    "undercooked",
    "raw",
    "stale",
    "rubbery",
    "salty",
    "too salty",
    "sweet",
    "too sweet",
    "watery",
    "mushy",
    "hard",
    "tough",
    "chewy",
    "mediocre",
    "average",
    "overpriced",
    "expensive",
    "tiny",
    "small",
    "slow",
    "meh",
    "gross",
    "nasty",
    "inedible",
    "lukewarm",
    "underwhelming",
}

NEGATIONS = {
    "not",
    "no",
    "never",
    "n't",
    "hardly",
    "barely",
    "without",
}

INTENSIFIERS = {
    "very",
    "really",
    "super",
    "extremely",
    "so",
    "too",
    "absolutely",
    "totally",
    "completely",
    "highly",
}

CONTRAST_MARKERS = {
    "but",
    "however",
    "although",
    "though",
    "yet",
    "nevertheless",
    "still",
}

POSITIVE_PHRASES = {
    "highly recommend",
    "would recommend",
    "cooked perfectly",
    "perfectly cooked",
    "melt in your mouth",
    "melted in my mouth",
    "to die for",
    "hit the spot",
    "full of flavor",
    "bursting with flavor",
}

NEGATIVE_PHRASES = {
    "not good",
    "not great",
    "not worth",
    "not impressed",
    "would not recommend",
    "was cold",
    "were cold",
    "was dry",
    "were dry",
    "too salty",
    "too sweet",
    "over cooked",
    "under cooked",
    "fell short",
}

print("Positive terms:", len(POSITIVE_LEXICON))
print("Negative terms:", len(NEGATIVE_LEXICON))
print("Positive phrases:", len(POSITIVE_PHRASES))
print("Negative phrases:", len(NEGATIVE_PHRASES))

Positive terms: 44
Negative terms: 46
Positive phrases: 10
Negative phrases: 14


In [36]:
# ============================================================
# 13. Tokenización ligera para scoring
# ============================================================

WORD_PATTERN = re.compile(r"\b[\w'-]+\b", flags=re.UNICODE)

def tokenize_words(text: str) -> list[str]:
    if not isinstance(text, str):
        return []

    return [
        match.group(0).lower()
        for match in WORD_PATTERN.finditer(text)
    ]


def count_phrase_matches(text: str, phrases: set[str]) -> int:
    if not isinstance(text, str):
        return 0

    text_lower = text.lower()
    count = 0

    for phrase in phrases:
        pattern = r"\b" + re.escape(phrase.lower()) + r"\b"
        count += len(re.findall(pattern, text_lower))

    return count


def has_contrast_marker(text: str) -> bool:
    tokens = tokenize_words(text)
    return any(token in CONTRAST_MARKERS for token in tokens)

In [37]:
# ============================================================
# 14. Función de scoring local
# ============================================================

def score_context_sentiment(text: str) -> dict:
    """
    Calcula señales de sentimiento en un contexto textual.
    Devuelve conteos positivos/negativos y score.
    """

    tokens = tokenize_words(text)

    positive_hits = []
    negative_hits = []

    for idx, token in enumerate(tokens):
        prev_tokens = tokens[max(0, idx - 3):idx]
        has_negation_before = any(prev in NEGATIONS for prev in prev_tokens)
        has_intensifier_before = any(prev in INTENSIFIERS for prev in prev_tokens)

        weight = 1.0
        if has_intensifier_before:
            weight += 0.5

        if token in POSITIVE_LEXICON:
            if has_negation_before:
                negative_hits.append({
                    "term": token,
                    "weight": weight,
                    "reason": "negated_positive",
                })
            else:
                positive_hits.append({
                    "term": token,
                    "weight": weight,
                    "reason": "positive_lexicon",
                })

        if token in NEGATIVE_LEXICON:
            if has_negation_before:
                positive_hits.append({
                    "term": token,
                    "weight": weight,
                    "reason": "negated_negative",
                })
            else:
                negative_hits.append({
                    "term": token,
                    "weight": weight,
                    "reason": "negative_lexicon",
                })

    positive_phrase_count = count_phrase_matches(text, POSITIVE_PHRASES)
    negative_phrase_count = count_phrase_matches(text, NEGATIVE_PHRASES)

    for _ in range(positive_phrase_count):
        positive_hits.append({
            "term": "positive_phrase",
            "weight": 1.75,
            "reason": "positive_phrase",
        })

    for _ in range(negative_phrase_count):
        negative_hits.append({
            "term": "negative_phrase",
            "weight": 1.75,
            "reason": "negative_phrase",
        })

    positive_score = sum(hit["weight"] for hit in positive_hits)
    negative_score = sum(hit["weight"] for hit in negative_hits)

    net_score = positive_score - negative_score

    return {
        "positive_score": float(positive_score),
        "negative_score": float(negative_score),
        "net_score": float(net_score),
        "positive_hits": positive_hits,
        "negative_hits": negative_hits,
        "has_contrast_marker": has_contrast_marker(text),
        "token_count": len(tokens),
    }

## 3. Asignación híbrida de sentimiento por mención

En este bloque se combinan varias señales:

1. Sentimiento local de la frase donde aparece el plato.
2. Sentimiento de la ventana cercana.
3. Señales del contexto izquierdo y derecho.
4. Rating/sentimiento general de la review como fallback.
5. Confianza del NER.
6. Flags de ambigüedad.

La salida principal será:

- `mention_sentiment_label_v1`
- `mention_sentiment_score_v1`
- `mention_sentiment_confidence_v1`
- `sentiment_reason_v1`
- `sentiment_flags_v1`

La etiqueta final se limitará a:

- `positive`
- `neutral`
- `negative`

Los casos mixtos o dudosos se marcarán mediante flags.

In [38]:
# ============================================================
# 15. Ajustes del léxico antes del scoring
# ============================================================

# Algunos términos son ambiguos y pueden generar falsos negativos si se usan solos.
# Los dejamos preferentemente como frases ("too sweet", "too small", etc.).
AMBIGUOUS_NEGATIVE_SINGLE_WORDS = {
    "sweet",
    "small",
    "hot",
}

NEGATIVE_LEXICON = NEGATIVE_LEXICON - AMBIGUOUS_NEGATIVE_SINGLE_WORDS

# Añadimos frases negativas más específicas.
NEGATIVE_PHRASES = NEGATIVE_PHRASES | {
    "too small",
    "too hot",
    "not fresh",
    "not crispy",
    "not flavorful",
    "lacked flavor",
    "lack of flavor",
    "no flavor",
    "nothing special",
}

POSITIVE_PHRASES = POSITIVE_PHRASES | {
    "very good",
    "really good",
    "so good",
    "super good",
    "pretty good",
    "quite good",
    "worth trying",
    "worth it",
    "came out hot",
    "served hot",
}

print("Positive lexicon:", len(POSITIVE_LEXICON))
print("Negative lexicon:", len(NEGATIVE_LEXICON))
print("Positive phrases:", len(POSITIVE_PHRASES))
print("Negative phrases:", len(NEGATIVE_PHRASES))

Positive lexicon: 44
Negative lexicon: 44
Positive phrases: 20
Negative phrases: 23


In [39]:
# ============================================================
# 16. Rating prior y helpers
# ============================================================

def rating_to_prior_score(rating_value) -> float:
    """
    Convierte rating general de review en una señal débil.
    No debe dominar el sentimiento local.
    """

    try:
        rating = float(rating_value)
    except Exception:
        return 0.0

    if rating >= 5:
        return 0.75
    if rating >= 4:
        return 0.45
    if rating == 3:
        return 0.0
    if rating <= 1:
        return -0.75
    if rating <= 2:
        return -0.45

    return 0.0


def review_sentiment_to_prior_score(sentiment_label: str) -> float:
    sentiment_label = str(sentiment_label).lower().strip()

    if sentiment_label == "positive":
        return 0.35
    if sentiment_label == "negative":
        return -0.35
    return 0.0


def flatten_hit_terms(hits: list[dict]) -> list[str]:
    terms = []

    for hit in hits:
        term = hit.get("term", "")
        reason = hit.get("reason", "")

        if reason:
            terms.append(f"{term}:{reason}")
        else:
            terms.append(term)

    return terms


def clipped_score(value: float, min_value: float = -5.0, max_value: float = 5.0) -> float:
    return float(max(min_value, min(max_value, value)))

In [40]:
# ============================================================
# 17. Scoring combinado por mención
# ============================================================

def compute_mention_sentiment_v1(row) -> dict:
    """
    Calcula el sentimiento híbrido v1 de una mención de plato.
    """

    sentence_context = row.get("sentence_context", "")
    window_context = row.get("window_context", "")
    left_context = row.get("left_context", "")
    right_context = row.get("right_context", "")

    sentence_score = score_context_sentiment(sentence_context)
    window_score = score_context_sentiment(window_context)
    left_score = score_context_sentiment(left_context)
    right_score = score_context_sentiment(right_context)

    rating_prior = rating_to_prior_score(row.get("rating_value"))
    review_prior = review_sentiment_to_prior_score(row.get("sentiment_label"))

    # La frase local manda. La ventana ayuda, pero pesa menos.
    local_net = (
        0.55 * sentence_score["net_score"]
        + 0.20 * window_score["net_score"]
        + 0.15 * right_score["net_score"]
        + 0.10 * left_score["net_score"]
    )

    prior_net = 0.70 * rating_prior + 0.30 * review_prior

    # El prior general solo ayuda cuando el contexto local es débil.
    local_signal_strength = (
        sentence_score["positive_score"]
        + sentence_score["negative_score"]
        + 0.5 * (
            right_score["positive_score"]
            + right_score["negative_score"]
            + left_score["positive_score"]
            + left_score["negative_score"]
        )
    )

    if local_signal_strength >= 1.0:
        combined_score = local_net + 0.15 * prior_net
        reason = "local_context_primary"
    else:
        combined_score = local_net + 0.65 * prior_net
        reason = "review_prior_fallback"

    combined_score = clipped_score(combined_score)

    total_positive = (
        sentence_score["positive_score"]
        + right_score["positive_score"]
        + left_score["positive_score"]
    )

    total_negative = (
        sentence_score["negative_score"]
        + right_score["negative_score"]
        + left_score["negative_score"]
    )

    has_positive = total_positive > 0
    has_negative = total_negative > 0

    flags = []

    if sentence_score["has_contrast_marker"] or window_score["has_contrast_marker"]:
        flags.append("contrast_marker_nearby")

    if has_positive and has_negative:
        flags.append("mixed_local_evidence")

    if local_signal_strength < 1.0:
        flags.append("weak_local_sentiment_signal")

    if reason == "review_prior_fallback":
        flags.append("used_review_prior_fallback")

    if row.get("confidence_mean", 1.0) < 0.80:
        flags.append("low_ner_confidence")

    if row.get("normalization_status_v2", "") != "normalized_rule_based_v2":
        flags.append("not_normalized_v2")

    # Etiquetado final
    if has_positive and has_negative and abs(combined_score) < 0.75:
        label = "neutral"
        flags.append("ambiguous_mixed_resolved_as_neutral")
    elif combined_score >= 0.65:
        label = "positive"
    elif combined_score <= -0.65:
        label = "negative"
    else:
        label = "neutral"

    # Confianza interpretable
    evidence_count = len(sentence_score["positive_hits"]) + len(sentence_score["negative_hits"])
    evidence_count += len(right_score["positive_hits"]) + len(right_score["negative_hits"])
    evidence_count += len(left_score["positive_hits"]) + len(left_score["negative_hits"])

    abs_score = abs(combined_score)

    if reason == "local_context_primary":
        confidence = 0.50 + min(0.30, abs_score * 0.10) + min(0.15, evidence_count * 0.04)
    else:
        confidence = 0.35 + min(0.20, abs_score * 0.08)

    if "mixed_local_evidence" in flags:
        confidence -= 0.10

    if "weak_local_sentiment_signal" in flags:
        confidence -= 0.08

    # Modulamos suavemente por confianza NER
    ner_confidence = row.get("confidence_mean", 1.0)
    try:
        ner_confidence = float(ner_confidence)
    except Exception:
        ner_confidence = 1.0

    confidence = confidence * (0.80 + 0.20 * ner_confidence)
    confidence = float(max(0.05, min(0.95, confidence)))

    positive_terms = []
    negative_terms = []

    for score_obj in [sentence_score, right_score, left_score]:
        positive_terms.extend(flatten_hit_terms(score_obj["positive_hits"]))
        negative_terms.extend(flatten_hit_terms(score_obj["negative_hits"]))

    return {
        "mention_sentiment_label_v1": label,
        "mention_sentiment_score_v1": float(combined_score),
        "mention_sentiment_confidence_v1": round(confidence, 4),
        "sentiment_reason_v1": reason,
        "sentiment_flags_v1": sorted(set(flags)),
        "local_signal_strength_v1": float(local_signal_strength),
        "rating_prior_score_v1": float(rating_prior),
        "review_prior_score_v1": float(review_prior),
        "sentence_positive_score_v1": float(sentence_score["positive_score"]),
        "sentence_negative_score_v1": float(sentence_score["negative_score"]),
        "window_positive_score_v1": float(window_score["positive_score"]),
        "window_negative_score_v1": float(window_score["negative_score"]),
        "positive_terms_v1": sorted(set(positive_terms)),
        "negative_terms_v1": sorted(set(negative_terms)),
    }

In [41]:
# ============================================================
# 18. Aplicar sentimiento híbrido v1
# ============================================================

sentiment_records = []

for row in tqdm(
    mentions_context_df.to_dict(orient="records"),
    desc="Calculando sentimiento por mención"
):
    sentiment_records.append(compute_mention_sentiment_v1(row))

sentiment_features_df = pd.DataFrame(sentiment_records)

mentions_sentiment_df = pd.concat(
    [
        mentions_context_df.reset_index(drop=True),
        sentiment_features_df.reset_index(drop=True)
    ],
    axis=1
)

print("Shape mentions_sentiment_df:", mentions_sentiment_df.shape)

print("\nDistribución de sentimiento por mención:")
display(mentions_sentiment_df["mention_sentiment_label_v1"].value_counts())

print("\nDistribución de reason:")
display(mentions_sentiment_df["sentiment_reason_v1"].value_counts())

print("\nConfianza sentimiento:")
display(mentions_sentiment_df["mention_sentiment_confidence_v1"].describe())

display(
    mentions_sentiment_df[
        [
            "dish_mention_text",
            "canonical_dish_name_v2",
            "rating_value",
            "sentiment_label",
            "mention_sentiment_label_v1",
            "mention_sentiment_score_v1",
            "mention_sentiment_confidence_v1",
            "sentiment_reason_v1",
            "sentiment_flags_v1",
            "positive_terms_v1",
            "negative_terms_v1",
            "sentence_context",
        ]
    ].head(20)
)

Calculando sentimiento por mención:   0%|          | 0/94932 [00:00<?, ?it/s]

Shape mentions_sentiment_df: (94932, 69)

Distribución de sentimiento por mención:


mention_sentiment_label_v1
positive    55325
neutral     31752
negative     7855
Name: count, dtype: int64


Distribución de reason:


sentiment_reason_v1
local_context_primary    66361
review_prior_fallback    28571
Name: count, dtype: int64


Confianza sentimiento:


count    94932.000000
mean         0.607725
std          0.223591
min          0.242200
25%          0.326600
50%          0.664900
75%          0.783600
max          0.949900
Name: mention_sentiment_confidence_v1, dtype: float64

,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,sentence_context
0,tamale,tamale,3.0,neutral,positive,1.2500,0.7446,local_context_primary,[],"[fresh:positive_lexicon, good:positive_lexicon]",[],"Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon."
1,lamb curry,lamb curry,5.0,positive,positive,1.5445,0.8039,local_context_primary,[],"[delicious:positive_lexicon, favorite:positive_lexicon, yummy:positive_lexicon]",[],Our favorite is the lamb curry and korma.
2,korma,korma,5.0,positive,positive,1.5445,0.8040,local_context_primary,[],"[delicious:positive_lexicon, favorite:positive_lexicon, yummy:positive_lexicon]",[],Our favorite is the lamb curry and korma.
3,naan,naan,5.0,positive,positive,0.9945,0.7193,local_context_primary,[],"[delicious:positive_lexicon, favorite:positive_lexicon, yummy:positive_lexicon]",[],With 10 different kinds of naan!!!
4,sandwiches,sandwich,4.0,positive,positive,1.3380,0.7537,local_context_primary,[],"[good:positive_lexicon, great:positive_lexicon, positive_phrase:positive_phrase]",[],"Really like that sandwiches come w salad, esp after eating too many curds!"
5,wings,wings,5.0,positive,positive,1.9945,0.8493,local_context_primary,[],"[amazing:positive_lexicon, delicious:positive_lexicon, perfectly:positive_lexicon, tender:positive_lexicon]",[],Amazingly amazing wings and homemade bleu cheese.
6,sushi,sushi,3.0,neutral,neutral,0.0000,0.2700,review_prior_fallback,"[used_review_prior_fallback, weak_local_sentiment_signal]",[],[],Our waitress brought our separate sushi orders on one plate so we couldn't really tell who's was who's and forgot several items on an order.
7,gnocchi with marinara,gnocchi with marinara,4.0,positive,positive,3.0380,0.8246,local_context_primary,"[contrast_marker_nearby, mixed_local_evidence]","[good:positive_lexicon, positive_phrase:positive_phrase]",[slow:negative_lexicon],"Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is just that good"
8,baked eggplant appetizer,baked eggplant appetizer,4.0,positive,positive,3.3880,0.8114,local_context_primary,"[contrast_marker_nearby, low_ner_confidence, mixed_local_evidence]","[good:positive_lexicon, positive_phrase:positive_phrase]",[slow:negative_lexicon],"Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is just that good"
9,Sonoran Dog,sonoran dog,4.0,positive,neutral,0.2730,0.2910,review_prior_fallback,"[used_review_prior_fallback, weak_local_sentiment_signal]",[],[],The bun makes the Sonoran Dog.


In [42]:
# ============================================================
# 19. QA del sentimiento por mención
# ============================================================

flag_counter = Counter()

for flags in mentions_sentiment_df["sentiment_flags_v1"]:
    if isinstance(flags, list):
        for flag in flags:
            flag_counter[flag] += 1

sentiment_qa_summary = {
    "total_mentions": int(len(mentions_sentiment_df)),
    "mention_sentiment_counts": {
        str(k): int(v)
        for k, v in mentions_sentiment_df["mention_sentiment_label_v1"].value_counts().to_dict().items()
    },
    "sentiment_reason_counts": {
        str(k): int(v)
        for k, v in mentions_sentiment_df["sentiment_reason_v1"].value_counts().to_dict().items()
    },
    "sentiment_flag_counts": {
        str(k): int(v)
        for k, v in flag_counter.items()
    },
    "confidence": {
        "min": float(mentions_sentiment_df["mention_sentiment_confidence_v1"].min()),
        "mean": float(mentions_sentiment_df["mention_sentiment_confidence_v1"].mean()),
        "median": float(mentions_sentiment_df["mention_sentiment_confidence_v1"].median()),
        "max": float(mentions_sentiment_df["mention_sentiment_confidence_v1"].max()),
    },
    "label_by_review_sentiment": (
        pd.crosstab(
            mentions_sentiment_df["sentiment_label"],
            mentions_sentiment_df["mention_sentiment_label_v1"]
        )
        .to_dict()
    ),
    "label_by_rating": (
        pd.crosstab(
            mentions_sentiment_df["rating_value"],
            mentions_sentiment_df["mention_sentiment_label_v1"]
        )
        .to_dict()
    ),
}

print(json.dumps(sentiment_qa_summary, indent=2, ensure_ascii=False)[:5000])

print("\nCrosstab review sentiment vs mention sentiment:")
display(
    pd.crosstab(
        mentions_sentiment_df["sentiment_label"],
        mentions_sentiment_df["mention_sentiment_label_v1"],
        margins=True
    )
)

print("\nCrosstab rating vs mention sentiment:")
display(
    pd.crosstab(
        mentions_sentiment_df["rating_value"],
        mentions_sentiment_df["mention_sentiment_label_v1"],
        margins=True
    )
)

print("\nFlags:")
display(
    pd.DataFrame(flag_counter.most_common(), columns=["flag", "count"])
)

{
  "total_mentions": 94932,
  "mention_sentiment_counts": {
    "positive": 55325,
    "neutral": 31752,
    "negative": 7855
  },
  "sentiment_reason_counts": {
    "local_context_primary": 66361,
    "review_prior_fallback": 28571
  },
  "sentiment_flag_counts": {
    "used_review_prior_fallback": 28571,
    "weak_local_sentiment_signal": 28571,
    "contrast_marker_nearby": 35113,
    "mixed_local_evidence": 14674,
    "low_ner_confidence": 4769,
    "ambiguous_mixed_resolved_as_neutral": 8161
  },
  "confidence": {
    "min": 0.2422,
    "mean": 0.607725492984452,
    "median": 0.6649,
    "max": 0.9499
  },
  "label_by_review_sentiment": {
    "negative": {
      "negative": 5527,
      "neutral": 1428,
      "positive": 900
    },
    "neutral": {
      "negative": 8700,
      "neutral": 7225,
      "positive": 15827
    },
    "positive": {
      "negative": 2168,
      "neutral": 5251,
      "positive": 47906
    }
  },
  "label_by_rating": {
    "negative": {
      "1.0": 309

mention_sentiment_label_v1,negative,neutral,positive,All
sentiment_label,,,,
negative,5527,8700,2168,16395
neutral,1428,7225,5251,13904
positive,900,15827,47906,64633
All,7855,31752,55325,94932



Crosstab rating vs mention sentiment:


mention_sentiment_label_v1,negative,neutral,positive,All
rating_value,,,,
1.0,3094,3595,674,7363
2.0,2433,5105,1494,9032
3.0,1428,7225,5251,13904
4.0,625,10357,18489,29471
5.0,275,5470,29417,35162
All,7855,31752,55325,94932



Flags:


,flag,count
0,contrast_marker_nearby,35113
1,used_review_prior_fallback,28571
2,weak_local_sentiment_signal,28571
3,mixed_local_evidence,14674
4,ambiguous_mixed_resolved_as_neutral,8161
5,low_ner_confidence,4769


In [43]:
# ============================================================
# 20. Revisión de casos interesantes
# ============================================================

review_cols = [
    "dish_mention_text",
    "canonical_dish_name_v2",
    "rating_value",
    "sentiment_label",
    "mention_sentiment_label_v1",
    "mention_sentiment_score_v1",
    "mention_sentiment_confidence_v1",
    "sentiment_reason_v1",
    "sentiment_flags_v1",
    "positive_terms_v1",
    "negative_terms_v1",
    "sentence_context",
    "window_context",
]

print("Casos positivos de alta confianza:")
display(
    mentions_sentiment_df[
        mentions_sentiment_df["mention_sentiment_label_v1"] == "positive"
    ]
    .sort_values("mention_sentiment_confidence_v1", ascending=False)
    [review_cols]
    .head(10)
)

print("\nCasos negativos de alta confianza:")
display(
    mentions_sentiment_df[
        mentions_sentiment_df["mention_sentiment_label_v1"] == "negative"
    ]
    .sort_values("mention_sentiment_confidence_v1", ascending=False)
    [review_cols]
    .head(10)
)

print("\nCasos neutrales / ambiguos:")
display(
    mentions_sentiment_df[
        mentions_sentiment_df["mention_sentiment_label_v1"] == "neutral"
    ]
    .sort_values("mention_sentiment_confidence_v1", ascending=True)
    [review_cols]
    .head(10)
)

print("\nCasos con sentimiento de mención distinto al sentimiento general de la review:")
display(
    mentions_sentiment_df[
        mentions_sentiment_df["mention_sentiment_label_v1"] != mentions_sentiment_df["sentiment_label"]
    ]
    .sort_values("mention_sentiment_confidence_v1", ascending=False)
    [review_cols]
    .head(20)
)

Casos positivos de alta confianza:


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,sentence_context,window_context
53625,pizza,pizza,2.0,negative,positive,3.0495,0.9499,local_context_primary,[contrast_marker_nearby],"[good:positive_lexicon, positive_phrase:positive_phrase]",[],The curds were very good but the pizza wasn't knock-your-socks-off like I was hoping.,pizza. We had the cheese curds as an appetizer and the Deep dish Capone for our meal. The curds were very good but the pizza wasn't knock-your-socks-off like I was hoping. The crust and cheese were good but the copious amounts of sauce and scan
42999,pizza,pizza,4.0,positive,positive,3.6755,0.9499,local_context_primary,[contrast_marker_nearby],"[delicious:positive_lexicon, good:positive_lexicon, positive_phrase:positive_phrase]",[],"Mostly Organic but all the way delicious, these pizzas this is very good pizza.","Mostly Organic but all the way delicious, these pizzas this is very good pizza. I eat gluten free and this was an unexpected gem. Strangely enough, I was on my way to have drinks with friends but I"
6879,Fries,fries,5.0,positive,positive,3.9945,0.9499,local_context_primary,[],"[amazing:positive_lexicon, fresh:positive_lexicon, good:positive_lexicon, perfect:positive_lexicon, seasoned:positive_lexicon]",[],"Fries seasoned just right, tartar sauce and the FISH the fresh, clean, battered perfect and meaty.","Where to start. The coleslaw was amazing and fresh good fresh crunch. Fries seasoned just right, tartar sauce and the FISH the fresh, clean, battered perfect and meaty."
7001,burger,burger,4.0,positive,positive,3.3505,0.9499,local_context_primary,[contrast_marker_nearby],"[good:positive_lexicon, great:positive_lexicon, positive_phrase:positive_phrase]",[],Pretty good burger and wings.,"Great local bar. Exremely good bartenders, they never let a drink get close to being empty. Pretty good burger and wings. Great prices, miss the 50-75-100 special though..."
7002,wings,wings,4.0,positive,positive,3.3505,0.9499,local_context_primary,[contrast_marker_nearby],"[good:positive_lexicon, great:positive_lexicon, positive_phrase:positive_phrase]",[],Pretty good burger and wings.,"Great local bar. Exremely good bartenders, they never let a drink get close to being empty. Pretty good burger and wings. Great prices, miss the 50-75-100 special though..."
7014,burgers,burger,4.0,positive,positive,4.1880,0.9499,local_context_primary,[],"[good:positive_lexicon, great:positive_lexicon, positive_phrase:positive_phrase]",[],"Great burgers, very good value for your money.","Great burgers, very good value for your money. Family friendly, good service, pretty clean. Will be returning."
53723,pulled pork,pulled pork,4.0,positive,positive,4.7380,0.9499,local_context_primary,[contrast_marker_nearby],"[fresh:positive_lexicon, good:positive_lexicon, positive_phrase:positive_phrase]",[],"So now the review 1) service, friendly and often we didn't have to wait long at all for our food or to be asked if we needed anything so service gets 5 stars 2) the menu is limited at lunchtime but both my food and my friends tasted fresh I just found it odd that the pulled pork had no sauce it was still good just not what I was expecting and t...",s 5 stars 2) the menu is limited at lunchtime but both my food and my friends tasted fresh I just found it odd that the pulled pork had no sauce it was still good just not what I was expecting and the homemade chips were really good too. We also had d
42845,french fries,fries,5.0,positive,positive,3.3820,0.9499,local_context_primary,[],"[delicious:positive_lexicon, positive_phrase:positive_phrase, recommend:positive_lexicon]",[],I HIGHLY recommend The Huey & their french fries.,", we ordered our food and once we tasted it we knew what all of the fuss was about. I HIGHLY recommend The Huey & their french fries. Absol


Casos negativos de alta confianza:


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,sentence_context,window_context
78593,crab,crab,1.0,negative,negative,-4.0195,0.9499,local_context_primary,[contrast_marker_nearby],[],"[dry:negative_lexicon, flavorless:negative_lexicon, negative_phrase:negative_phrase, stale:negative_lexicon]","The crab & artichoke dip was only okay, and the flatbread it came with was dry, flavorless, and had the unappealing texture of stale bread.","e menu looked promising. But the preparation, presentation, and the flavors all fell far short of our expectations. The crab & artichoke dip was only okay, and the flatbread it came with was dry, flavorless, and had the unappealing texture of s"
40191,ribs,ribs,4.0,positive,negative,-3.5495,0.9499,local_context_primary,[contrast_marker_nearby],[],"[dry:negative_lexicon, negative_phrase:negative_phrase, tough:negative_lexicon]","The ribs left a lot to be desired, they were dry and very tough, I had trouble cutting through the ribs with my knife, I would not order them again.","ould eat them again. The ribs left a lot to be desired, they were dry and very tough, I had trouble cutting through the ribs with my knife, I would not order them again. I didn't have high expectations on the brisket after trying the ribs but m"
21714,fries,fries,1.0,negative,negative,-3.4695,0.9499,local_context_primary,[contrast_marker_nearby],[],"[burnt:negative_lexicon, cold:negative_lexicon, negative_phrase:negative_phrase]","When I got my order, the fries were cold, burnt, and was filled halfway.","rience I've had with Hardee's at this location. I ordered three burgers and a large curly fry. When I got my order, the fries were cold, burnt, and was filled halfway. I'm not all about getting every last bite but if I ordered a large order of f"
21717,burgers,burger,1.0,negative,negative,-5.0000,0.9499,local_context_primary,[contrast_marker_nearby],[],"[cold:negative_lexicon, dry:negative_lexicon, hard:negative_lexicon, horrible:negative_lexicon, negative_phrase:negative_phrase]",The burgers were cold and those new pretzel buns were horrible and hard.,"t all about getting every last bite but if I ordered a large order of fries, please give me a large order of fries. The burgers were cold and those new pretzel buns were horrible and hard. They were dry and the bottom bun wasn't a full bottom bun."
6916,fries,fries,1.0,negative,negative,-3.4695,0.9499,local_context_primary,[],[],"[disappointed:negative_lexicon, negative_phrase:negative_phrase, worth:negated_positive]","No parking, specials were not explained in full after asking, had to go get the waitress for the fries (tried the Parmesan fries and disappointed to see a pinch of parm, not worth), had to find ketchup ourselves, had to get up and ask for a new beer and the tab.","upstairs. Just saying. No parking, specials were not explained in full after asking, had to go get the waitress for the fries (tried the Parmesan fries and disappointed to see a pinch of parm, not worth), had to find ketchup ourselves, had to ge"
77706,pizza,pizza,2.0,negative,negative,-4.1380,0.9499,local_context_primary,[],[],"[cold:negative_lexicon, expensive:negative_lexicon, nasty:negative_lexicon, negative_phrase:negative_phrase, soggy:negative_lexicon]",Got pizza from here it was cold and soggy.,Got pizza from here it was cold and soggy. The crust tasted like pure garlic nasty. I will never go there again. To expensive for
77255,shrimp,shrimp,1.0,negative,negative,-4.5445,0.9499,local_context_primary,[contrast_marker_nearby],[],"[bad:negative_lexicon, chewy:negative_lexicon, flavorless:negative_lexicon, overcooked:negative_lexicon, worst:negative_lexicon]","The scallops were just ok (not pan seared), the pesto sauce wasn't bad but had a different taste that I can't quite explain, but the shrimp were horribly overcoo


Casos neutrales / ambiguos:


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,sentence_context,window_context
22424,chicken,chicken,3.0,neutral,neutral,0.0,0.2422,review_prior_fallback,"[contrast_marker_nearby, low_ner_confidence, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],The bruschetta chicken salad was pretty goo but needed more tomatoes...,(not sure we needed that). He did an efficient job managing our orders and kept up with all our refills. The bruschetta chicken salad was pretty goo but needed more tomatoes...yes I know it was a bruschetta salad with minimal tomatoes. Also blue c
86183,salad,salad,3.0,neutral,neutral,0.0,0.2428,review_prior_fallback,"[low_ner_confidence, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],My advice is to give it a try and hopefully they read this and go to work on their cole slaw and potato salad and make their side item pricing reflect a true upgrade in price and quality.,ere just right. My advice is to give it a try and hopefully they read this and go to work on their cole slaw and potato salad and make their side item pricing reflect a true upgrade in price and quality. They also need more signs to clearly iden
72092,#,#,3.0,neutral,neutral,0.0,0.2430,review_prior_fallback,"[low_ner_confidence, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],"When I went here, I ordered a #15.","When I went here, I ordered a #15. Pho Dac Biet: rice noodle soup with rare beef, beef brisket and beef meat balls ($8.95). For the price, it's a littl"
65361,tea,tea,3.0,neutral,neutral,0.0,0.2435,review_prior_fallback,"[low_ner_confidence, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],Not sure if they're using a powdered tea for that too?,le powder stuff. I got a regular milk tea and something just wasn't right with it. Not sure if they're using a powdered tea for that too? It didn't taste like brewed tea for sure and I make a better milk tea at home with trader joe's unsweeten
19259,Thai,thai,3.0,neutral,neutral,0.0,0.2437,review_prior_fallback,"[low_ner_confidence, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],The Thai Chicken Pizza served up by the student had nice flavor with the peanut sauce drizzled over it.,"he pizza in a timely fashion. FOOD: B Fortunately for the student, Mr. Mark A is not terribly picky with his pizza. The Thai Chicken Pizza served up by the student had nice flavor with the peanut sauce drizzled over it. There was a nice proport"
39988,coffee,coffee,3.0,neutral,neutral,0.0,0.2439,review_prior_fallback,"[low_ner_confidence, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],Came here to try a frozen coffee since it was in the area we were shopping.,Came here to try a frozen coffee since it was in the area we were shopping. I got the mocha frappe which was okay. I wouldn't drive distances for it. Th
8975,jerk,jerk,3.0,neutral,neutral,0.0,0.2439,review_prior_fallback,"[low_ner_confidence, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],"That all being said, I'm planning on going to the live reggae shows they have and ordering take-out when the urge for jerk chicken arises.",". That all being said, I'm planning on going to the live reggae shows they have and ordering take-out when the urge for jerk chicken arises."
72157,smoothies,smoothy,3.0,neutral,neutral,0.0,0.2439,review_prior_fallback,"[contrast_marker_nearby, low_ner_confidence, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],"I'm guessing it will always be as such but worse with time because this is a one stop shop hodge podge of wings, seafood, sandwiches, smoothies, and gyros.","t will always be as such but worse with time because this is a one stop shop hodge podge of wings, seafood, sandwiches, smoothies, and gyros. Next, we approach the counte


Casos con sentimiento de mención distinto al sentimiento general de la review:


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,sentence_context,window_context
87064,tacos,tacos,3.0,neutral,positive,3.3750,0.9499,local_context_primary,[contrast_marker_nearby],"[good:positive_lexicon, positive_phrase:positive_phrase]",[],"My boyfriend's tacos were good, I'm not denying that the food was pretty good, but really?","the salmon, I would have been REALLY annoyed at paying $15 for a very small beet and goat cheese salad. My boyfriend's tacos were good, I'm not denying that the food was pretty good, but really? $16 for 3 tacos that just came with a few tortill"
4953,Sushi,sushi,3.0,neutral,negative,-3.0000,0.9499,local_context_primary,[contrast_marker_nearby],[],"[great:negated_positive, negative_phrase:negative_phrase, slow:negative_lexicon]",Sushi is okay but not great.,ver even there was a lot of table. This bar is so dark. The table was still sticky and that sauce plate was also dirty. Sushi is okay but not great. I don't like the service and the atmospheres in the restaurant. Everything was so slow and it li
20652,Pulled pork,pulled pork,3.0,neutral,positive,3.3125,0.9499,local_context_primary,[],"[best:positive_lexicon, good:positive_lexicon, great:positive_lexicon, positive_phrase:positive_phrase]",[],Pulled pork wasn't the best I've ever had.,Decor and theme of the restaurant is great. We had the nachos appetizer - it was really good. Pulled pork wasn't the best I've ever had. Sides were really good. Kind of on the pricey side for bbq.
20072,burgers,burger,3.0,neutral,negative,-3.0000,0.9499,local_context_primary,[],[],"[cold:negative_lexicon, hard:negative_lexicon, lukewarm:negative_lexicon, slow:negative_lexicon]",the service was slow and inattentive since all three -- the burgers and the fires came out lukewarm to almost cold.,"h the time was the only consideration in the 3 star rating. the service was slow and inattentive since all three -- the burgers and the fires came out lukewarm to almost cold. we ordered medium cooked burgers and the bison was well done, hard and"
20767,enchiladas,enchilada,3.0,neutral,positive,3.0000,0.9499,local_context_primary,[contrast_marker_nearby],"[creamy:positive_lexicon, delicious:positive_lexicon, favorite:positive_lexicon, good:positive_lexicon]",[],The enchiladas with pork are delicious and the creamy chipotle chicken has become a family favorite.,e gallo with house made tortilla chips. It's not easy to find a chile relleno around here and they make a good one. The enchiladas with pork are delicious and the creamy chipotle chicken has become a family favorite. We haven't tried their drinks yet
45553,crab legs,crab leg,3.0,neutral,positive,5.0000,0.9499,local_context_primary,[],"[enjoyed:positive_lexicon, good:positive_lexicon, hot:positive_lexicon, positive_phrase:positive_phrase, worth:positive_lexicon]",[],"really enjoyed their lobster fettucine, mushroom ravioli, clam chowder, crab legs, gelato was really really good and the cherry pie.","to say I feel like the $20.99 wasn't worth it. really enjoyed their lobster fettucine, mushroom ravioli, clam chowder, crab legs, gelato was really really good and the cherry pie. They should have some kind of hot sauce like tapatio for their cevic"
46342,fried chicken,fried chicken,3.0,neutral,positive,4.0500,0.9499,local_context_primary,[],"[good:positive_lexicon, positive_phrase:positive_phrase, tasty:positive_lexicon, worth:positive_lexicon]",[],I got the fried chicken that was on the menu it was pretty good.,I got the fried chicken that was on the menu it was pretty good. Very tasty. Worth 21 dollars? I don't think so. Probably worth 15 or something
75840,wings,wings,3.0,neutral,positive,3.3625,0.9499,local_context_primary,[],"[crisp:positive_lexicon, dry:negated_negative, good:positive_lexicon, positive_phrase:positive_phrase, tasty:positive_

## 4. Refinamiento aspect-based v1.1

La primera asignación híbrida funciona, pero se ha detectado un problema importante: algunas palabras positivas o negativas de la frase pueden referirse a otro plato o a otro aspecto de la experiencia.

Ejemplo:

```text
The curds were very good but the pizza wasn't knock-your-socks-off.

```
En este caso, very good no describe pizza, sino curds.

Para mejorar esto, se añade una versión refinada v1.1 que prioriza:

cláusula concreta donde aparece el plato;
contexto inmediatamente cercano a la mención;
patrones directos como pizza was delicious, burger was dry, fries were cold;
uso más limitado del rating general;
flags de ambigüedad.

In [44]:
# ============================================================
# 21. Extraer cláusula objetivo de la mención
# ============================================================

CLAUSE_PUNCTUATION_PATTERN = re.compile(r"[.!?;]")
CONTRAST_BOUNDARY_PATTERN = re.compile(
    r"\b(but|however|although|though|yet|nevertheless)\b",
    flags=re.IGNORECASE
)

def extract_sentence_with_offsets(text: str, start_char: int, end_char: int) -> dict:
    """
    Extrae frase aproximada y conserva offsets dentro del texto completo.
    """

    if not isinstance(text, str):
        return {
            "sentence_text": "",
            "sentence_start": 0,
            "sentence_end": 0,
        }

    text_len = len(text)

    start_char = max(0, min(int(start_char), text_len))
    end_char = max(0, min(int(end_char), text_len))

    left_text = text[:start_char]
    right_text = text[end_char:]

    left_boundaries = list(SENTENCE_BOUNDARY_PATTERN.finditer(left_text))

    if left_boundaries:
        sentence_start = left_boundaries[-1].end()
    else:
        sentence_start = 0

    right_boundary = SENTENCE_BOUNDARY_PATTERN.search(right_text)

    if right_boundary:
        sentence_end = end_char + right_boundary.end()
    else:
        sentence_end = text_len

    sentence_text = normalize_space(text[sentence_start:sentence_end])

    return {
        "sentence_text": sentence_text,
        "sentence_start": int(sentence_start),
        "sentence_end": int(sentence_end),
    }


def extract_target_clause(text: str, start_char: int, end_char: int) -> str:
    """
    Extrae la cláusula más cercana a la mención.
    Usa puntuación fuerte y conectores de contraste como límites.
    """

    if not isinstance(text, str):
        return ""

    sentence_info = extract_sentence_with_offsets(text, start_char, end_char)

    sentence_text_raw = text[
        sentence_info["sentence_start"]:sentence_info["sentence_end"]
    ]

    rel_start = int(start_char) - sentence_info["sentence_start"]
    rel_end = int(end_char) - sentence_info["sentence_start"]

    rel_start = max(0, min(rel_start, len(sentence_text_raw)))
    rel_end = max(0, min(rel_end, len(sentence_text_raw)))

    boundaries = [0, len(sentence_text_raw)]

    for match in CLAUSE_PUNCTUATION_PATTERN.finditer(sentence_text_raw):
        boundaries.append(match.end())

    for match in CONTRAST_BOUNDARY_PATTERN.finditer(sentence_text_raw):
        # Si el contraste aparece antes de la mención, la cláusula suele empezar después.
        boundaries.append(match.end())

        # Si aparece después, la cláusula suele terminar antes.
        boundaries.append(match.start())

    boundaries = sorted(set(boundaries))

    clause_start = 0
    clause_end = len(sentence_text_raw)

    for boundary in boundaries:
        if boundary <= rel_start:
            clause_start = max(clause_start, boundary)

        if boundary >= rel_end:
            clause_end = min(clause_end, boundary)

    clause = sentence_text_raw[clause_start:clause_end]

    return normalize_space(clause)


def extract_near_mention_context(text: str, start_char: int, end_char: int, left_chars: int = 45, right_chars: int = 90) -> str:
    """
    Extrae un contexto mucho más cercano a la mención.
    """

    if not isinstance(text, str):
        return ""

    text_len = len(text)

    start_char = max(0, min(int(start_char), text_len))
    end_char = max(0, min(int(end_char), text_len))

    left_start = max(0, start_char - left_chars)
    right_end = min(text_len, end_char + right_chars)

    return normalize_space(text[left_start:right_end])

In [45]:
# ============================================================
# 22. Patrones directos de sentimiento sobre la mención
# ============================================================

DIRECT_POSITIVE_PATTERNS = [
    r"\b(was|were|is|are)\s+(very\s+|really\s+|so\s+|super\s+|pretty\s+|quite\s+)?(good|great|amazing|awesome|excellent|fantastic|delicious|tasty|fresh|crispy|tender|juicy|flavorful|perfect|perfectly cooked)\b",
    r"\b(loved|love|enjoyed|enjoy|recommend|recommended)\b",
    r"\b(worth trying|worth it|hit the spot|to die for|full of flavor)\b",
]

DIRECT_NEGATIVE_PATTERNS = [
    r"\b(was|were|is|are)\s+(very\s+|really\s+|so\s+|super\s+|pretty\s+|quite\s+|too\s+)?(bad|awful|terrible|horrible|bland|cold|dry|soggy|burnt|overcooked|undercooked|raw|stale|rubbery|salty|greasy|oily|tasteless|flavorless|mediocre|disappointing|underwhelming|tough|chewy)\b",
    r"\b(wasn'?t|was not|weren'?t|were not|isn'?t|is not|aren'?t|are not)\s+(that\s+|very\s+|really\s+|as\s+)?(good|great|best|amazing|awesome|excellent|fantastic|delicious|tasty|fresh|crispy|flavorful|worth)\b",
    r"\b(not good|not great|not worth|not impressed|nothing special|only okay|just okay|left a lot to be desired|would not order|wouldn'?t order|would not recommend|wouldn'?t recommend)\b",
]

def count_regex_patterns(text: str, patterns: list[str]) -> int:
    if not isinstance(text, str):
        return 0

    count = 0
    text_lower = text.lower()

    for pattern in patterns:
        count += len(re.findall(pattern, text_lower))

    return count


def score_direct_patterns(text: str) -> dict:
    positive_direct = count_regex_patterns(text, DIRECT_POSITIVE_PATTERNS)
    negative_direct = count_regex_patterns(text, DIRECT_NEGATIVE_PATTERNS)

    return {
        "direct_positive_count": int(positive_direct),
        "direct_negative_count": int(negative_direct),
        "direct_pattern_score": float(1.25 * positive_direct - 1.25 * negative_direct),
    }

In [46]:
# ============================================================
# 23. Crear contextos target-focused
# ============================================================

target_context_records = []

for row in tqdm(
    mentions_sentiment_df.to_dict(orient="records"),
    desc="Extrayendo contextos target-focused"
):
    start_char = safe_int(row.get("start_char"), 0)
    end_char = safe_int(row.get("end_char"), start_char)

    target_clause = extract_target_clause(
        text=row.get("review_text", ""),
        start_char=start_char,
        end_char=end_char,
    )

    near_context = extract_near_mention_context(
        text=row.get("review_text", ""),
        start_char=start_char,
        end_char=end_char,
        left_chars=45,
        right_chars=90,
    )

    target_context_records.append({
        "target_clause_context_v11": target_clause,
        "near_mention_context_v11": near_context,
    })

target_context_df = pd.DataFrame(target_context_records)

mentions_sentiment_v11_df = pd.concat(
    [
        mentions_sentiment_df.reset_index(drop=True),
        target_context_df.reset_index(drop=True),
    ],
    axis=1
)

print("Shape mentions_sentiment_v11_df:", mentions_sentiment_v11_df.shape)

display(
    mentions_sentiment_v11_df[
        [
            "dish_mention_text",
            "canonical_dish_name_v2",
            "mention_sentiment_label_v1",
            "sentence_context",
            "target_clause_context_v11",
            "near_mention_context_v11",
        ]
    ].head(15)
)

Extrayendo contextos target-focused:   0%|          | 0/94932 [00:00<?, ?it/s]

Shape mentions_sentiment_v11_df: (94932, 71)


,dish_mention_text,canonical_dish_name_v2,mention_sentiment_label_v1,sentence_context,target_clause_context_v11,near_mention_context_v11
0,tamale,tamale,positive,"Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","rtment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has"
1,lamb curry,lamb curry,positive,Our favorite is the lamb curry and korma.,Our favorite is the lamb curry and korma.,"y, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we"
2,korma,korma,positive,Our favorite is the lamb curry and korma.,Our favorite is the lamb curry and korma.,elicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost ch
3,naan,naan,positive,With 10 different kinds of naan!!!,With 10 different kinds of naan!,curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try
4,sandwiches,sandwich,positive,"Really like that sandwiches come w salad, esp after eating too many curds!","Really like that sandwiches come w salad, esp after eating too many curds!","very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. W"
5,wings,wings,positive,Amazingly amazing wings and homemade bleu cheese.,Amazingly amazing wings and homemade bleu cheese.,"Amazingly amazing wings and homemade bleu cheese. Had the ribeye: tender, perfectly prepared, delicious. Nice sel"
6,sushi,sushi,neutral,Our waitress brought our separate sushi orders on one plate so we couldn't really tell who's was who's and forgot several items on an order.,Our waitress brought our separate sushi orders on one plate so we couldn't really tell who's was who's and forgot several items on an order.,r hibachi. Our waitress brought our separate sushi orders on one plate so we couldn't really tell who's was who's and forgot several items o
7,gnocchi with marinara,gnocchi with marinara,positive,"Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is just that good","Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow,","Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite thi"
8,baked eggplant appetizer,baked eggplant appetizer,positive,"Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is just that good","Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow,","od food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow, but despite this, I'd go back, the food is j"
9,Sonoran Dog,sonoran dog,neutral,The bun makes the Sonoran Dog.,The bun makes the Sonoran Dog.,"The bun makes the Sonoran Dog. It's like a snuggie for the pup. A first, it seems ridiculous and almost like it's going"


In [47]:
# ============================================================
# 24. Scoring refinado v1.1
# ============================================================

def compute_mention_sentiment_v11(row) -> dict:
    """
    Refinamiento v1.1: más centrado en la cláusula y contexto inmediato de la mención.
    """

    target_clause = row.get("target_clause_context_v11", "")
    near_context = row.get("near_mention_context_v11", "")
    sentence_context = row.get("sentence_context", "")

    clause_score = score_context_sentiment(target_clause)
    near_score = score_context_sentiment(near_context)
    sentence_score = score_context_sentiment(sentence_context)
    direct_score = score_direct_patterns(target_clause)

    rating_prior = rating_to_prior_score(row.get("rating_value"))
    review_prior = review_sentiment_to_prior_score(row.get("sentiment_label"))

    # Priorizamos cláusula y contexto cercano.
    target_net = (
        0.55 * clause_score["net_score"]
        + 0.25 * near_score["net_score"]
        + 0.10 * sentence_score["net_score"]
        + direct_score["direct_pattern_score"]
    )

    prior_net = 0.70 * rating_prior + 0.30 * review_prior

    target_signal_strength = (
        clause_score["positive_score"]
        + clause_score["negative_score"]
        + direct_score["direct_positive_count"]
        + direct_score["direct_negative_count"]
        + 0.40 * (near_score["positive_score"] + near_score["negative_score"])
    )

    flags = []

    if clause_score["has_contrast_marker"] or near_score["has_contrast_marker"]:
        flags.append("contrast_marker_nearby")

    if direct_score["direct_positive_count"] > 0:
        flags.append("direct_positive_pattern")

    if direct_score["direct_negative_count"] > 0:
        flags.append("direct_negative_pattern")

    if target_signal_strength < 1.0:
        flags.append("weak_target_sentiment_signal")

    if row.get("confidence_mean", 1.0) < 0.80:
        flags.append("low_ner_confidence")

    # El prior solo pesa si no hay señales target suficientes.
    if target_signal_strength >= 1.0:
        combined_score = target_net + 0.10 * prior_net
        reason = "target_context_primary"
    else:
        combined_score = target_net + 0.55 * prior_net
        reason = "review_prior_fallback"
        flags.append("used_review_prior_fallback")

    combined_score = clipped_score(combined_score)

    target_positive = clause_score["positive_score"] + near_score["positive_score"]
    target_negative = clause_score["negative_score"] + near_score["negative_score"]

    if target_positive > 0 and target_negative > 0:
        flags.append("mixed_target_evidence")

    # Reglas de decisión
    if (
        target_positive > 0
        and target_negative > 0
        and abs(combined_score) < 0.80
    ):
        label = "neutral"
        flags.append("ambiguous_mixed_resolved_as_neutral")

    elif combined_score >= 0.70:
        label = "positive"

    elif combined_score <= -0.70:
        label = "negative"

    else:
        label = "neutral"

    # Confianza
    abs_score = abs(combined_score)

    if reason == "target_context_primary":
        confidence = 0.52 + min(0.28, abs_score * 0.10) + min(0.12, target_signal_strength * 0.04)
    else:
        confidence = 0.34 + min(0.18, abs_score * 0.08)

    if "mixed_target_evidence" in flags:
        confidence -= 0.08

    if "weak_target_sentiment_signal" in flags:
        confidence -= 0.08

    if "direct_positive_pattern" in flags or "direct_negative_pattern" in flags:
        confidence += 0.05

    ner_confidence = row.get("confidence_mean", 1.0)

    try:
        ner_confidence = float(ner_confidence)
    except Exception:
        ner_confidence = 1.0

    confidence = confidence * (0.80 + 0.20 * ner_confidence)
    confidence = float(max(0.05, min(0.95, confidence)))

    positive_terms = []
    negative_terms = []

    for score_obj in [clause_score, near_score]:
        positive_terms.extend(flatten_hit_terms(score_obj["positive_hits"]))
        negative_terms.extend(flatten_hit_terms(score_obj["negative_hits"]))

    return {
        "mention_sentiment_label_v11": label,
        "mention_sentiment_score_v11": float(combined_score),
        "mention_sentiment_confidence_v11": round(confidence, 4),
        "sentiment_reason_v11": reason,
        "sentiment_flags_v11": sorted(set(flags)),
        "target_signal_strength_v11": float(target_signal_strength),
        "rating_prior_score_v11": float(rating_prior),
        "review_prior_score_v11": float(review_prior),
        "target_clause_positive_score_v11": float(clause_score["positive_score"]),
        "target_clause_negative_score_v11": float(clause_score["negative_score"]),
        "near_context_positive_score_v11": float(near_score["positive_score"]),
        "near_context_negative_score_v11": float(near_score["negative_score"]),
        "direct_positive_count_v11": int(direct_score["direct_positive_count"]),
        "direct_negative_count_v11": int(direct_score["direct_negative_count"]),
        "positive_terms_v11": sorted(set(positive_terms)),
        "negative_terms_v11": sorted(set(negative_terms)),
    }

In [48]:
# ============================================================
# 25. Aplicar scoring v1.1
# ============================================================

sentiment_v11_records = []

for row in tqdm(
    mentions_sentiment_v11_df.to_dict(orient="records"),
    desc="Calculando sentimiento refinado v1.1"
):
    sentiment_v11_records.append(compute_mention_sentiment_v11(row))

sentiment_v11_features_df = pd.DataFrame(sentiment_v11_records)

mentions_sentiment_v11_df = pd.concat(
    [
        mentions_sentiment_v11_df.reset_index(drop=True),
        sentiment_v11_features_df.reset_index(drop=True)
    ],
    axis=1
)

print("Distribución sentimiento v1:")
display(mentions_sentiment_v11_df["mention_sentiment_label_v1"].value_counts())

print("\nDistribución sentimiento v1.1:")
display(mentions_sentiment_v11_df["mention_sentiment_label_v11"].value_counts())

print("\nCambios v1 -> v1.1:")
display(
    pd.crosstab(
        mentions_sentiment_v11_df["mention_sentiment_label_v1"],
        mentions_sentiment_v11_df["mention_sentiment_label_v11"],
        margins=True
    )
)

print("\nReasons v1.1:")
display(mentions_sentiment_v11_df["sentiment_reason_v11"].value_counts())

print("\nConfianza v1.1:")
display(mentions_sentiment_v11_df["mention_sentiment_confidence_v11"].describe())

Calculando sentimiento refinado v1.1:   0%|          | 0/94932 [00:00<?, ?it/s]

Distribución sentimiento v1:


mention_sentiment_label_v1
positive    55325
neutral     31752
negative     7855
Name: count, dtype: int64


Distribución sentimiento v1.1:


mention_sentiment_label_v11
neutral     45366
positive    43142
negative     6424
Name: count, dtype: int64

...

mention_sentiment_label_v11,negative,neutral,positive,All
mention_sentiment_label_v1,,,,
negative,5335,2430,90,7855
neutral,1043,28930,1779,31752
positive,46,14006,41273,55325
All,6424,45366,43142,94932



Reasons v1.1:


sentiment_reason_v11
target_context_primary    50956
review_prior_fallback     43976
Name: count, dtype: int64


Confianza v1.1:


count    94932.000000
mean         0.551181
std          0.264267
min          0.166000
25%          0.287600
50%          0.614100
75%          0.795200
max          0.950000
Name: mention_sentiment_confidence_v11, dtype: float64

In [49]:
# ============================================================
# 26. Revisar casos cambiados por v1.1
# ============================================================

review_cols_v11 = [
    "dish_mention_text",
    "canonical_dish_name_v2",
    "rating_value",
    "sentiment_label",
    "mention_sentiment_label_v1",
    "mention_sentiment_score_v1",
    "mention_sentiment_label_v11",
    "mention_sentiment_score_v11",
    "mention_sentiment_confidence_v11",
    "sentiment_reason_v11",
    "sentiment_flags_v11",
    "positive_terms_v11",
    "negative_terms_v11",
    "target_clause_context_v11",
    "near_mention_context_v11",
    "sentence_context",
]

changed_sentiment_df = mentions_sentiment_v11_df[
    mentions_sentiment_v11_df["mention_sentiment_label_v1"]
    != mentions_sentiment_v11_df["mention_sentiment_label_v11"]
].copy()

print("Total cambios de etiqueta:", len(changed_sentiment_df))

print("\nCambios positive -> neutral/negative:")
display(
    changed_sentiment_df[
        changed_sentiment_df["mention_sentiment_label_v1"] == "positive"
    ]
    .sort_values("mention_sentiment_confidence_v11", ascending=False)
    [review_cols_v11]
    .head(20)
)

print("\nCambios negative -> neutral/positive:")
display(
    changed_sentiment_df[
        changed_sentiment_df["mention_sentiment_label_v1"] == "negative"
    ]
    .sort_values("mention_sentiment_confidence_v11", ascending=False)
    [review_cols_v11]
    .head(20)
)

print("\nCasos v1.1 positivos alta confianza:")
display(
    mentions_sentiment_v11_df[
        mentions_sentiment_v11_df["mention_sentiment_label_v11"] == "positive"
    ]
    .sort_values("mention_sentiment_confidence_v11", ascending=False)
    [review_cols_v11]
    .head(10)
)

print("\nCasos v1.1 negativos alta confianza:")
display(
    mentions_sentiment_v11_df[
        mentions_sentiment_v11_df["mention_sentiment_label_v11"] == "negative"
    ]
    .sort_values("mention_sentiment_confidence_v11", ascending=False)
    [review_cols_v11]
    .head(10)
)

Total cambios de etiqueta: 19394

Cambios positive -> neutral/negative:


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_label_v11,mention_sentiment_score_v11,mention_sentiment_confidence_v11,sentiment_reason_v11,sentiment_flags_v11,positive_terms_v11,negative_terms_v11,target_clause_context_v11,near_mention_context_v11,sentence_context
36426,French fries,fries,4.0,positive,positive,0.8130,negative,-3.7955,0.8899,target_context_primary,"[contrast_marker_nearby, direct_negative_pattern, mixed_target_evidence]","[good:positive_lexicon, positive_phrase:positive_phrase]","[good:negated_positive, negative_phrase:negative_phrase]","The French fries were not good,","I experience sand in a few of my bites. The French fries were not good, but the Onion rings were very good!!. We wanted to try the Boston Cream pi","The French fries were not good, but the Onion rings were very good!!."
91225,ribs,ribs,2.0,negative,positive,1.2995,negative,-3.1545,0.8899,target_context_primary,"[contrast_marker_nearby, direct_negative_pattern, mixed_target_evidence]",[good:positive_lexicon],"[negative_phrase:negative_phrase, tough:negative_lexicon]","The gumbo was so-so nothing special, the red beans and rice tasted like refried beans, the greens tasted like they came out of a can, the jambalaya wasn't good either, the fried chicken was fried too long, the ribs were ok","r, the fried chicken was fried too long, the ribs were ok but were a little tough, the Mac n cheese was so-so, the fried catfish was actual","The gumbo was so-so nothing special, the red beans and rice tasted like refried beans, the greens tasted like they came out of a can, the jambalaya wasn't good either, the fried chicken was fried too long, the ribs were ok but were a little tough, the Mac n cheese was so-so, the fried catfish was actually very good and the bread pudding was hor..."
24207,crab cake,crab cakes,1.0,negative,positive,0.7555,negative,-2.0130,0.8672,target_context_primary,[direct_negative_pattern],[],[disappointing:negative_lexicon],it was disappointing waiting 40 minutes for one crab cake .,was disappointing waiting 40 minutes for one crab cake ... Our main course arrived about 15 minute after the crab cake. I ordered the shrimp and,"The crab cake was actually good and savory, but it was disappointing waiting 40 minutes for one crab cake ..."
10634,tempura,tempura,1.0,negative,positive,0.7555,negative,-2.0130,0.8672,target_context_primary,[direct_negative_pattern],[],[horrible:negative_lexicon],the owner Wes is horrible I came in Last week to order tempura chicken they changed there Menu so she said get a child one for 7 I was fine with that it came with a lot of stuff which I don't eat I just eat the tempura so they gave me Just Chicken on a plate it used to be 3.,Wes is horrible I came in Last week to order tempura chicken they changed there Menu so she said get a child one for 7 I was fine with that it,The food is good love the spicy mayo the employee are nice but the owner Wes is horrible I came in Last week to order tempura chicken they changed there Menu so she said get a child one for 7 I was fine with that it came with a lot of stuff which I don't eat I just eat the tempura so they gave me Just Chicken on a plate it used to be 3.
51050,steak,steak,5.0,positive,positive,0.7945,negative,-1.7245,0.7824,target_context_primary,"[direct_negative_pattern, mixed_target_evidence]","[good:positive_lexicon, positive_phrase:positive_phrase]",[chewy:negative_lexicon],Our only complaint was that the steak was very chewy.,perfection. Our only complaint was that the steak was very chewy. The enchiladas were very good and the stuffed poblano was bursting with f,Our only complaint was that the steak was very chewy.
70990,lobster,lobster,1.0,negative,positive,1.2430,negative,-1.7005,0.7800,target_context_primary,"[contrast_marker_nearby, direct_negative_pattern, mixed_target_evidence]","[good:positive_lexicon, positive_phrase:positive_phrase]",[salty:negative_


Cambios negative -> neutral/positive:


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_label_v11,mention_sentiment_score_v11,mention_sentiment_confidence_v11,sentiment_reason_v11,sentiment_flags_v11,positive_terms_v11,negative_terms_v11,target_clause_context_v11,near_mention_context_v11,sentence_context
31747,burger,burger,4.0,positive,negative,-1.0620,positive,2.2670,0.9070,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern]","[excellent:positive_lexicon, good:positive_lexicon]",[],The burger was excellent (,"r, a few beers, and the brussel sprouts. The burger was excellent (though it was too huge for me to finish) and the brussel sprouts were good",The burger was excellent (though it was too huge for me to finish) and the brussel sprouts were good (though not great).
66128,quesadilla,quesadilla,3.0,neutral,negative,-1.4875,positive,2.1250,0.8942,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern]","[great:positive_lexicon, love:positive_lexicon]",[],I love the cheese and chorizo quesadilla.,"ot great), but I love the cheese and chorizo quesadilla. I think CJ Muggs does most of its business from catering events, so their dining room is","I've found that pretty much everything on the menu is okay (not great), but I love the cheese and chorizo quesadilla."
9760,crab,crab,3.0,neutral,negative,-0.7750,positive,2.8375,0.8899,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern, mixed_target_evidence]","[great:positive_lexicon, love:positive_lexicon]","[good:negated_positive, negative_phrase:negative_phrase]",The appetizers were great I love the spring rolls and the dynamite shrimp and also the crab wontons,g rolls and the dynamite shrimp and also the crab wontons but the sesame chicken was not good at all like it didn't taste like nothin I wou,The appetizers were great I love the spring rolls and the dynamite shrimp and also the crab wontons but the sesame chicken was not good at all like it didn't taste like nothin I would eat again.
14823,steak,steak,2.0,negative,negative,-1.1380,positive,3.2205,0.8899,target_context_primary,"[direct_positive_pattern, mixed_target_evidence]","[crispy:positive_lexicon, enjoyed:positive_lexicon, good:positive_lexicon]","[enjoy:negated_positive, hard:negative_lexicon, negative_phrase:negative_phrase]",TRY: Crispy black striped bass (All my friends got the steak except me and they did NOT enjoy it as much as i enjoyed my sea bass~ They said it was hard to chew and lacked flavor - order BIG RED OWL BURGER instead if you want to have meat cause i heard their burger is good.,y black striped bass (All my friends got the steak except me and they did NOT enjoy it as much as i enjoyed my sea bass~ They said it was ha,TRY: Crispy black striped bass (All my friends got the steak except me and they did NOT enjoy it as much as i enjoyed my sea bass~ They said it was hard to chew and lacked flavor - order BIG RED OWL BURGER instead if you want to have meat cause i heard their burger is good....
9758,spring rolls,spring rolls,3.0,neutral,negative,-0.7750,positive,3.3375,0.8899,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern, mixed_target_evidence]","[great:positive_lexicon, love:positive_lexicon]","[good:negated_positive, negative_phrase:negative_phrase]",The appetizers were great I love the spring rolls and the dynamite shrimp and also the crab wontons,The appetizers were great I love the spring rolls and the dynamite shrimp and also the crab wontons but the sesame chicken was not good at,The appetizers were great I love the spring rolls and the dynamite shrimp and also the crab wontons but the sesame chicken was not good at all like it didn't taste like nothin I would eat again.
9759,shrimp,shrimp,3.0,neutral,negative,-0.7750,positive,3.0875,0.8894,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern, mixed_target_evidence]","[great:positive_lexico


Casos v1.1 positivos alta confianza:


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_label_v11,mention_sentiment_score_v11,mention_sentiment_confidence_v11,sentiment_reason_v11,sentiment_flags_v11,positive_terms_v11,negative_terms_v11,target_clause_context_v11,near_mention_context_v11,sentence_context
16,burgers,burger,3.0,neutral,positive,2.7625,positive,4.175,0.95,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern]","[good:positive_lexicon, positive_phrase:positive_phrase]",[],"All of their beers are very good, and I am also a fan of their burgers and tenderloins.","are very good, and I am also a fan of their burgers and tenderloins. Therefore, I was excited to try their pizza, but I don't think it ended","All of their beers are very good, and I am also a fan of their burgers and tenderloins."
94878,oysters,oysters,4.0,positive,positive,3.5755,positive,5.000,0.95,target_context_primary,[direct_positive_pattern],"[loved:positive_lexicon, perfectly:positive_lexicon, positive_phrase:positive_phrase]",[],Loved loved loved them oysters.,"hem in the slightest. Loved loved loved them oysters. The steamer basket was also very well prepared. All seafood cooked perfectly, not even r",Loved loved loved them oysters.
94861,calamari,calamari,2.0,negative,positive,1.6870,positive,3.008,0.95,target_context_primary,[direct_positive_pattern],[good:positive_lexicon],[],Maybe drinks some bar apps and you'll be good (calamari was good) .,ybe drinks some bar apps and you'll be good (calamari was good) . View was gorgeous.,Maybe drinks some bar apps and you'll be good (calamari was good) .
94858,shrimp,shrimp,2.0,negative,positive,2.1870,positive,3.458,0.95,target_context_primary,[direct_positive_pattern],"[fresh:positive_lexicon, great:positive_lexicon]",[],I will say that the shrimp was great and very very fresh tasting.,"ra crab or extra shrimp. I will say that the shrimp was great and very very fresh tasting. The crab, well, it was Alaskan legs. Could have be",I will say that the shrimp was great and very very fresh tasting.
74,wings,wings,5.0,positive,positive,3.7695,positive,4.938,0.95,target_context_primary,[direct_positive_pattern],"[awesome:positive_lexicon, good:positive_lexicon, hot:positive_lexicon, positive_phrase:positive_phrase]",[],I also ordered some hot wings and they were pretty good.,et one deal they do. I also ordered some hot wings and they were pretty good. They had a ton of sauce on them which is awesome. Next time I,I also ordered some hot wings and they were pretty good.
72,Shrimp Bisque the wife ordered,shrimp bisque the wife ordered,5.0,positive,positive,4.2195,positive,5.000,0.95,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern]","[amazing:positive_lexicon, positive_phrase:positive_phrase, recommend:positive_lexicon]",[],", the Shrimp Bisque the wife ordered was amazing, so I would recommend that going forward.","As much as I liked the chowder however, the Shrimp Bisque the wife ordered was amazing, so I would recommend that going forward. But enough of prattling on about si","As much as I liked the chowder however, the Shrimp Bisque the wife ordered was amazing, so I would recommend that going forward."
31,brisket,brisket,4.0,positive,positive,2.3630,positive,3.192,0.95,target_context_primary,[direct_positive_pattern],"[awesome:positive_lexicon, good:positive_lexicon, juicy:positive_lexicon, tender:positive_lexicon]",[],Beef brisket sandwich was awesome.,Went for lunch. Beef brisket sandwich was awesome. So juicy and tender. Pulled pork was was just as good!,Beef brisket sandwich was awesome.
94839,burrito,burrito,5.0,positive,positive,2.4195,positive,3.088,0.95,target_context_primary,[direct_positive_pattern],"[awesome:positive_lexicon, great:positive_lexicon, hot:positive_lexicon]",[],The breakfast burrito is great too.,"got her hot, both were great. The breakfast burrito is great too. Awesome decor and friendly 


Casos v1.1 negativos alta confianza:


,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_label_v11,mention_sentiment_score_v11,mention_sentiment_confidence_v11,sentiment_reason_v11,sentiment_flags_v11,positive_terms_v11,negative_terms_v11,target_clause_context_v11,near_mention_context_v11,sentence_context
94619,burger,burger,3.0,neutral,negative,-1.5000,negative,-3.7250,0.95,target_context_primary,[direct_negative_pattern],[],"[negative_phrase:negative_phrase, worth:negated_positive]",I will skip the burger next time not worth the high price,heir service is exceptional. I will skip the burger next time not worth the high price,I will skip the burger next time not worth the high price
56158,pizza,pizza,1.0,negative,negative,-3.7320,negative,-2.7505,0.95,target_context_primary,"[direct_negative_pattern, direct_positive_pattern]",[],"[cold:negative_lexicon, negative_phrase:negative_phrase, recommend:negated_positive]",Would not recommend to order for delivery unless you like cold pizza.,d to order for delivery unless you like cold pizza.,Would not recommend to order for delivery unless you like cold pizza.
55857,pizza,pizza,1.0,negative,negative,-3.1695,negative,-3.5630,0.95,target_context_primary,[direct_negative_pattern],[],[tough:negative_lexicon],"The crust on the pizza is tough, very very tough, and the sauce is more of a salsa than a Italian tomato sauce.","isgusting amount of garlic. The crust on the pizza is tough, very very tough, and the sauce is more of a salsa than a Italian tomato sauce.","The crust on the pizza is tough, very very tough, and the sauce is more of a salsa than a Italian tomato sauce."
499,naan,naan,1.0,negative,negative,-4.1820,negative,-5.0000,0.95,target_context_primary,[direct_negative_pattern],[],"[great:negated_positive, mediocre:negative_lexicon, negative_phrase:negative_phrase, tough:negative_lexicon]","Sorry to say this curry was not great, the sauce was very heavy on the salt and tomato and the goat was tough as an old boot, the naan bread was below mediocre.","o and the goat was tough as an old boot, the naan bread was below mediocre. Only nice thing was the rice. Very Very dissapointed","Sorry to say this curry was not great, the sauce was very heavy on the salt and tomato and the goat was tough as an old boot, the naan bread was below mediocre."
585,pizza,pizza,1.0,negative,negative,-2.7445,negative,-4.0130,0.95,target_context_primary,[direct_negative_pattern],[],"[cold:negative_lexicon, horrible:negative_lexicon, soggy:negative_lexicon]","Horrible experience, pizza was soggy and cold when I received it, took 2 hours until I actually received my order as well.","Horrible experience, pizza was soggy and cold when I received it, took 2 hours until I actually received my order as","Horrible experience, pizza was soggy and cold when I received it, took 2 hours until I actually received my order as well."
94523,Mussels,mussels,3.0,neutral,negative,-1.8000,negative,-3.0500,0.95,target_context_primary,[direct_negative_pattern],[],"[awful:negative_lexicon, dry:negative_lexicon]","Mussels are awful dry and not pleasant to look at, they almost tasted like jerky.","enu), this is the only roll I will eat here. Mussels are awful dry and not pleasant to look at, they almost tasted like jerky. I prefer the on","Mussels are awful dry and not pleasant to look at, they almost tasted like jerky."
56007,fried chicken,fried chicken,2.0,negative,negative,-1.7130,negative,-2.8920,0.95,target_context_primary,[direct_negative_pattern],[],"[salty:negative_lexicon, tiny:negative_lexicon]",My husband's fried chicken was so salty we had to send it back.,"nsane prices for tiny portions. My husband's fried chicken was so salty we had to send it back. We both left hungry, dehydrated, and poorer. This pl",My husband's fried chicken was so salty we had to send it back.
13296,burrito,burrito,1.0,negative,negative,-1.7945,negative,-2.9130,0.95,target_context_primary,"[con

## 5. Guardado final de sentimiento híbrido v1

Tras comparar la versión inicial `v1` y el refinamiento `v1.1`, se decide usar `v1.1` como salida principal del módulo híbrido.

La versión `v1.1` es más conservadora y está más centrada en el contexto real de la mención del plato.

Además de la etiqueta final, se crea un nivel de fiabilidad:

- `high`: etiqueta más fiable.
- `medium`: etiqueta útil, pero con cierta ambigüedad.
- `low`: etiqueta débil o dependiente del prior de la review.

Estos niveles serán importantes para los siguientes módulos de agregación y ranking.

In [50]:
# ============================================================
# 27. Crear columnas finales de sentimiento por mención
# ============================================================

mentions_final_df = mentions_sentiment_v11_df.copy()

# Usamos v1.1 como salida final del módulo híbrido.
mentions_final_df["mention_sentiment_label_final"] = mentions_final_df["mention_sentiment_label_v11"]
mentions_final_df["mention_sentiment_score_final"] = mentions_final_df["mention_sentiment_score_v11"]
mentions_final_df["mention_sentiment_confidence_final"] = mentions_final_df["mention_sentiment_confidence_v11"]
mentions_final_df["sentiment_reason_final"] = mentions_final_df["sentiment_reason_v11"]
mentions_final_df["sentiment_flags_final"] = mentions_final_df["sentiment_flags_v11"]
mentions_final_df["sentiment_method_final"] = "hybrid_context_lexicon_v1_1"


def compute_sentiment_reliability_tier(row) -> str:
    """
    Clasifica la fiabilidad de la etiqueta de sentimiento por mención.
    """

    confidence = float(row.get("mention_sentiment_confidence_final", 0.0))
    reason = row.get("sentiment_reason_final", "")
    flags = row.get("sentiment_flags_final", [])

    if not isinstance(flags, list):
        flags = []

    risky_flags = {
        "low_ner_confidence",
        "weak_target_sentiment_signal",
        "used_review_prior_fallback",
    }

    ambiguity_flags = {
        "mixed_target_evidence",
        "ambiguous_mixed_resolved_as_neutral",
        "contrast_marker_nearby",
    }

    has_risky_flag = any(flag in risky_flags for flag in flags)
    has_ambiguity_flag = any(flag in ambiguity_flags for flag in flags)

    if (
        confidence >= 0.75
        and reason == "target_context_primary"
        and not has_risky_flag
    ):
        return "high"

    if (
        confidence >= 0.50
        and not has_risky_flag
    ):
        return "medium"

    if confidence >= 0.60 and reason == "target_context_primary":
        return "medium"

    return "low"


def is_training_candidate(row) -> bool:
    """
    Marca menciones útiles para un futuro dataset weak-supervised
    de entrenamiento de aspect-based sentiment.
    """

    tier = row.get("sentiment_reliability_tier", "")
    label = row.get("mention_sentiment_label_final", "")
    reason = row.get("sentiment_reason_final", "")
    flags = row.get("sentiment_flags_final", [])

    if not isinstance(flags, list):
        flags = []

    if label not in {"positive", "negative", "neutral"}:
        return False

    if tier != "high":
        return False

    if reason != "target_context_primary":
        return False

    forbidden_flags = {
        "low_ner_confidence",
        "weak_target_sentiment_signal",
        "used_review_prior_fallback",
        "ambiguous_mixed_resolved_as_neutral",
    }

    if any(flag in forbidden_flags for flag in flags):
        return False

    return True


mentions_final_df["sentiment_reliability_tier"] = mentions_final_df.apply(
    compute_sentiment_reliability_tier,
    axis=1
)

mentions_final_df["is_sentiment_training_candidate"] = mentions_final_df.apply(
    is_training_candidate,
    axis=1
)

print("Distribución etiqueta final:")
display(mentions_final_df["mention_sentiment_label_final"].value_counts())

print("\nDistribución reliability tier:")
display(mentions_final_df["sentiment_reliability_tier"].value_counts())

print("\nCandidatas para futuro entrenamiento:")
display(mentions_final_df["is_sentiment_training_candidate"].value_counts())

display(
    mentions_final_df[
        [
            "dish_mention_text",
            "canonical_dish_name_v2",
            "rating_value",
            "sentiment_label",
            "mention_sentiment_label_final",
            "mention_sentiment_score_final",
            "mention_sentiment_confidence_final",
            "sentiment_reliability_tier",
            "is_sentiment_training_candidate",
            "sentiment_reason_final",
            "sentiment_flags_final",
            "target_clause_context_v11",
        ]
    ].head(20)
)

Distribución etiqueta final:


mention_sentiment_label_final
neutral     45366
positive    43142
negative     6424
Name: count, dtype: int64


Distribución reliability tier:


sentiment_reliability_tier
low       44426
medium    25495
high      25011
Name: count, dtype: int64


Candidatas para futuro entrenamiento:


is_sentiment_training_candidate
False    69921
True     25011
Name: count, dtype: int64

,dish_mention_text,canonical_dish_name_v2,rating_value,sentiment_label,mention_sentiment_label_final,mention_sentiment_score_final,mention_sentiment_confidence_final,sentiment_reliability_tier,is_sentiment_training_candidate,sentiment_reason_final,sentiment_flags_final,target_clause_context_v11
0,tamale,tamale,3.0,neutral,positive,1.1500,0.7066,medium,False,target_context_primary,[],"Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon."
1,lamb curry,lamb curry,5.0,positive,positive,1.2130,0.7128,medium,False,target_context_primary,[],Our favorite is the lamb curry and korma.
2,korma,korma,5.0,positive,positive,0.9630,0.6720,medium,False,target_context_primary,[],Our favorite is the lamb curry and korma.
3,naan,naan,5.0,positive,neutral,0.3465,0.2876,low,False,review_prior_fallback,"[used_review_prior_fallback, weak_target_sentiment_signal]",With 10 different kinds of naan!
4,sandwiches,sandwich,4.0,positive,positive,0.8545,0.6574,medium,False,target_context_primary,[],"Really like that sandwiches come w salad, esp after eating too many curds!"
5,wings,wings,5.0,positive,positive,1.7130,0.7952,high,True,target_context_primary,[],Amazingly amazing wings and homemade bleu cheese.
6,sushi,sushi,3.0,neutral,neutral,0.0000,0.2600,low,False,review_prior_fallback,"[used_review_prior_fallback, weak_target_sentiment_signal]",Our waitress brought our separate sushi orders on one plate so we couldn't really tell who's was who's and forgot several items on an order.
7,gnocchi with marinara,gnocchi with marinara,4.0,positive,positive,5.0000,0.8634,high,True,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern, mixed_target_evidence]","Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow,"
8,baked eggplant appetizer,baked eggplant appetizer,4.0,positive,positive,4.8670,0.8496,medium,False,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern, low_ner_confidence, mixed_target_evidence]","Good food--loved the gnocchi with marinara the baked eggplant appetizer was very good too The service was very slow,"
9,Sonoran Dog,sonoran dog,4.0,positive,neutral,0.2310,0.2777,low,False,review_prior_fallback,"[used_review_prior_fallback, weak_target_sentiment_signal]",The bun makes the Sonoran Dog.


In [51]:
# ============================================================
# 28. QA final del sentimiento híbrido
# ============================================================

final_flag_counter = Counter()

for flags in mentions_final_df["sentiment_flags_final"]:
    if isinstance(flags, list):
        for flag in flags:
            final_flag_counter[flag] += 1

label_counts = mentions_final_df["mention_sentiment_label_final"].value_counts()
tier_counts = mentions_final_df["sentiment_reliability_tier"].value_counts()
reason_counts = mentions_final_df["sentiment_reason_final"].value_counts()

final_sentiment_qa = {
    "total_mentions": int(len(mentions_final_df)),

    "mention_sentiment_counts": {
        str(k): int(v)
        for k, v in label_counts.to_dict().items()
    },

    "sentiment_reliability_tier_counts": {
        str(k): int(v)
        for k, v in tier_counts.to_dict().items()
    },

    "sentiment_reason_counts": {
        str(k): int(v)
        for k, v in reason_counts.to_dict().items()
    },

    "sentiment_flag_counts": {
        str(k): int(v)
        for k, v in final_flag_counter.items()
    },

    "training_candidate_counts": {
        str(k): int(v)
        for k, v in mentions_final_df["is_sentiment_training_candidate"].value_counts().to_dict().items()
    },

    "confidence": {
        "min": float(mentions_final_df["mention_sentiment_confidence_final"].min()),
        "mean": float(mentions_final_df["mention_sentiment_confidence_final"].mean()),
        "median": float(mentions_final_df["mention_sentiment_confidence_final"].median()),
        "max": float(mentions_final_df["mention_sentiment_confidence_final"].max()),
    },

    "label_by_review_sentiment": {
        str(label): {
            str(k): int(v)
            for k, v in values.items()
        }
        for label, values in pd.crosstab(
            mentions_final_df["sentiment_label"],
            mentions_final_df["mention_sentiment_label_final"]
        ).to_dict().items()
    },

    "label_by_rating": {
        str(label): {
            str(k): int(v)
            for k, v in values.items()
        }
        for label, values in pd.crosstab(
            mentions_final_df["rating_value"],
            mentions_final_df["mention_sentiment_label_final"]
        ).to_dict().items()
    },

    "label_by_reliability": {
        str(label): {
            str(k): int(v)
            for k, v in values.items()
        }
        for label, values in pd.crosstab(
            mentions_final_df["sentiment_reliability_tier"],
            mentions_final_df["mention_sentiment_label_final"]
        ).to_dict().items()
    },
}

print(json.dumps(final_sentiment_qa, indent=2, ensure_ascii=False)[:6000])

print("\nCrosstab rating vs mention sentiment final:")
display(
    pd.crosstab(
        mentions_final_df["rating_value"],
        mentions_final_df["mention_sentiment_label_final"],
        margins=True
    )
)

print("\nCrosstab reliability vs mention sentiment final:")
display(
    pd.crosstab(
        mentions_final_df["sentiment_reliability_tier"],
        mentions_final_df["mention_sentiment_label_final"],
        margins=True
    )
)

print("\nFlags finales:")
display(
    pd.DataFrame(final_flag_counter.most_common(), columns=["flag", "count"])
)

{
  "total_mentions": 94932,
  "mention_sentiment_counts": {
    "neutral": 45366,
    "positive": 43142,
    "negative": 6424
  },
  "sentiment_reliability_tier_counts": {
    "low": 44426,
    "medium": 25495,
    "high": 25011
  },
  "sentiment_reason_counts": {
    "target_context_primary": 50956,
    "review_prior_fallback": 43976
  },
  "sentiment_flag_counts": {
    "used_review_prior_fallback": 43976,
    "weak_target_sentiment_signal": 43976,
    "contrast_marker_nearby": 22594,
    "direct_positive_pattern": 17883,
    "mixed_target_evidence": 8166,
    "low_ner_confidence": 4769,
    "ambiguous_mixed_resolved_as_neutral": 4068,
    "direct_negative_pattern": 2365
  },
  "training_candidate_counts": {
    "False": 69921,
    "True": 25011
  },
  "confidence": {
    "min": 0.166,
    "mean": 0.5511806840685965,
    "median": 0.6141,
    "max": 0.95
  },
  "label_by_review_sentiment": {
    "negative": {
      "negative": 4076,
      "neutral": 1249,
      "positive": 1099
    

mention_sentiment_label_final,negative,neutral,positive,All
rating_value,,,,
1.0,2184,4452,727,7363
2.0,1892,5675,1465,9032
3.0,1249,8246,4409,13904
4.0,726,13769,14976,29471
5.0,373,13224,21565,35162
All,6424,45366,43142,94932



Crosstab reliability vs mention sentiment final:


mention_sentiment_label_final,negative,neutral,positive,All
sentiment_reliability_tier,,,,
high,2689,0,22322,25011
low,345,40990,3091,44426
medium,3390,4376,17729,25495
All,6424,45366,43142,94932



Flags finales:


,flag,count
0,used_review_prior_fallback,43976
1,weak_target_sentiment_signal,43976
2,contrast_marker_nearby,22594
3,direct_positive_pattern,17883
4,mixed_target_evidence,8166
5,low_ner_confidence,4769
6,ambiguous_mixed_resolved_as_neutral,4068
7,direct_negative_pattern,2365


In [52]:
# ============================================================
# 29. Crear muestras de revisión manual
# ============================================================

sample_cols = [
    "mention_id",
    "review_id",
    "business_id",
    "dish_mention_text",
    "canonical_dish_name_v2",
    "dish_id_v2",
    "rating_value",
    "sentiment_label",
    "mention_sentiment_label_final",
    "mention_sentiment_score_final",
    "mention_sentiment_confidence_final",
    "sentiment_reliability_tier",
    "sentiment_reason_final",
    "sentiment_flags_final",
    "positive_terms_v11",
    "negative_terms_v11",
    "target_clause_context_v11",
    "near_mention_context_v11",
    "sentence_context",
    "review_text",
]

# Muestra aleatoria general
sample_random_df = mentions_final_df.sample(
    n=min(300, len(mentions_final_df)),
    random_state=RANDOM_STATE
)[sample_cols].copy()

# Muestra de alta confianza
sample_high_conf_df = (
    mentions_final_df[
        mentions_final_df["sentiment_reliability_tier"] == "high"
    ]
    .sample(
        n=min(300, (mentions_final_df["sentiment_reliability_tier"] == "high").sum()),
        random_state=RANDOM_STATE
    )
    [sample_cols]
    .copy()
)

# Muestra de baja confianza / prior fallback
sample_low_conf_df = (
    mentions_final_df[
        (mentions_final_df["sentiment_reliability_tier"] == "low")
        | (mentions_final_df["sentiment_reason_final"] == "review_prior_fallback")
    ]
    .sort_values("mention_sentiment_confidence_final", ascending=True)
    .head(400)
    [sample_cols]
    .copy()
)

# Muestra de contradicción entre review general y mención
sample_disagreement_df = (
    mentions_final_df[
        mentions_final_df["mention_sentiment_label_final"]
        != mentions_final_df["sentiment_label"]
    ]
    .sort_values("mention_sentiment_confidence_final", ascending=False)
    .head(400)
    [sample_cols]
    .copy()
)

# Muestra de casos con evidencia mixta
sample_mixed_df = (
    mentions_final_df[
        mentions_final_df["sentiment_flags_final"].apply(
            lambda flags: isinstance(flags, list) and "mixed_target_evidence" in flags
        )
    ]
    .sort_values("mention_sentiment_confidence_final", ascending=False)
    .head(400)
    [sample_cols]
    .copy()
)

# Dataset candidato para entrenamiento futuro
sentiment_training_candidates_df = mentions_final_df[
    mentions_final_df["is_sentiment_training_candidate"]
].copy()

sentiment_training_candidates_df = sentiment_training_candidates_df[
    [
        "mention_id",
        "review_id",
        "business_id",
        "dish_mention_text",
        "canonical_dish_name_v2",
        "dish_id_v2",
        "target_clause_context_v11",
        "near_mention_context_v11",
        "sentence_context",
        "review_text",
        "rating_value",
        "sentiment_label",
        "mention_sentiment_label_final",
        "mention_sentiment_score_final",
        "mention_sentiment_confidence_final",
        "sentiment_reliability_tier",
        "sentiment_reason_final",
        "sentiment_flags_final",
        "positive_terms_v11",
        "negative_terms_v11",
    ]
].copy()

print("sample_random_df:", len(sample_random_df))
print("sample_high_conf_df:", len(sample_high_conf_df))
print("sample_low_conf_df:", len(sample_low_conf_df))
print("sample_disagreement_df:", len(sample_disagreement_df))
print("sample_mixed_df:", len(sample_mixed_df))
print("sentiment_training_candidates_df:", len(sentiment_training_candidates_df))

display(sample_random_df.head(5))
display(sample_high_conf_df.head(5))
display(sample_low_conf_df.head(5))
display(sample_disagreement_df.head(5))

sample_random_df: 300
sample_high_conf_df: 300
sample_low_conf_df: 400
sample_disagreement_df: 400
sample_mixed_df: 400
sentiment_training_candidates_df: 25011


,mention_id,review_id,business_id,dish_mention_text,canonical_dish_name_v2,dish_id_v2,rating_value,sentiment_label,mention_sentiment_label_final,mention_sentiment_score_final,mention_sentiment_confidence_final,sentiment_reliability_tier,sentiment_reason_final,sentiment_flags_final,positive_terms_v11,negative_terms_v11,target_clause_context_v11,near_mention_context_v11,sentence_context,review_text
34721,dish_mention_5db986d1e2f7d721ec91,otdAWbS2ayIwA_nxnjc1SQ,joeRm-_7T0XTkMdKo2nbaA,pizza,pizza,dish_seed_v2_000001,5.0,positive,positive,1.713,0.7951,high,target_context_primary,[],"[best:positive_lexicon, favorite:positive_lexicon, fresh:positive_lexicon, great:positive_lexicon]",[],Their Margarita pizza is the best!,"My favorite place for pizza. Their Margarita pizza is the best! With a great crust, sauce, fresh mozzarella cheese and basil. The real taste",Their Margarita pizza is the best!,"My favorite place for pizza. Their Margarita pizza is the best! With a great crust, sauce, fresh mozzarella cheese and basil. The real taste of Italian pizza. They also make great white pizza, with garlic spinach and mushrooms. The owners are from Italy. Very friendly staff and good service. It is a family owned place and Paula is wonderful! We..."
65182,dish_mention_1f99cb583768b44bcda6,hen92ss9F_jP9RHm3PnXHQ,jVdYRED2iztNaNCoTAhVMA,bang bang shrimp wrap,bang bang shrimp wrap,dish_seed_v2_001996,5.0,positive,positive,4.738,0.9479,high,target_context_primary,[direct_positive_pattern],"[delicious:positive_lexicon, good:positive_lexicon, great:positive_lexicon, positive_phrase:positive_phrase]",[],The bang bang shrimp wrap was so good!,ilable if you want customize your order. The bang bang shrimp wrap was so good! Also had the soup of the day that was delicious. Overall great selection of,The bang bang shrimp wrap was so good!,"I heard great this about this place so I decided to give it a try... So happy I did. I have a new place to stop in for a quick meal. Overall really good, casual dinning food. You order at the counter and everything is made in front of you. The food is fresh and delicious. Menu has something for everyone. Salads, wraps, paninis, soup. And build ..."
69970,dish_mention_98f1e6c35d604fb83b5f,IZyv573-PPf7UOE47FjBMw,TttFjRQ-8Iz8by4hsD7iOQ,salmon,salmon,dish_seed_v2_000017,5.0,positive,positive,2.463,0.9282,high,target_context_primary,[direct_positive_pattern],"[enjoyed:positive_lexicon, incredible:positive_lexicon]",[],"My family enjoyed the salad, salmon, eggplant, spaghetti special, and tiramisu.","in Center City. My family enjoyed the salad, salmon, eggplant, spaghetti special, and tiramisu. That homemade tiramisu is incredible and gene","My family enjoyed the salad, salmon, eggplant, spaghetti special, and tiramisu.","I kind of don't want anyone else to know about Angelina's, but this review is a testament to the great experience I had there. This intimate cash-only BYOB restaurant looks like it occupies a former single-family home. There are only a few tables, and it is a small operation. I live in the neighborhood and when I walk by, I often see Angela her..."
18439,dish_mention_348b10ec10bf7eca320e,joIF7xU0US9vfr5XGQFSkw,z7em5co2qckbAXoDGXynsA,Duck Fried rice,duck fried rice,dish_seed_v2_004220,5.0,positive,positive,1.213,0.6788,medium,target_context_primary,[low_ner_confidence],"[great:positive_lexicon, incredible:positive_lexicon]",[],The Duck Fried rice was incredible.,"ing menu and everything was so tasteful. The Duck Fried rice was incredible. Dan was a great server, very attentive. Portions are generous and despite",The Duck Fried rice was incredible.,"This place is amazing! Did the tasting menu and everything was so tasteful. The Duck Fried rice was incredible. Dan was a great server, very attentive. Portions are generous and despite the menu note of small, medium or large. So impresses"
65359,dish_mention_20ac5b8b118a32e8ebe6,nUwyJYhwfhPnmIMLS2BMrg,KYFJ7qh7bYYAsjnGrhH3NA,taro,taro,dish_seed_v2_000476,3.0,neutral,

,mention_id,review_id,business_id,dish_mention_text,canonical_dish_name_v2,dish_id_v2,rating_value,sentiment_label,mention_sentiment_label_final,mention_sentiment_score_final,mention_sentiment_confidence_final,sentiment_reliability_tier,sentiment_reason_final,sentiment_flags_final,positive_terms_v11,negative_terms_v11,target_clause_context_v11,near_mention_context_v11,sentence_context,review_text
10446,dish_mention_cb6488aad3d1d56764a8,rlg9uTNGxtdm2iOvGThR8Q,ICqgjbOpBD9SUtE5PQC9sA,Oysters,oysters,dish_seed_v2_000011,4.0,positive,positive,1.9295,0.7529,high,target_context_primary,"[contrast_marker_nearby, mixed_target_evidence]","[good:positive_lexicon, positive_phrase:positive_phrase]",[good:negated_positive],"Oysters good, lobster bisque good","ters, crab cake sandwich and lobster bisque. Oysters good, lobster bisque good but the crab cake sandwich not so good. The potatoes and slaw w","Oysters good, lobster bisque good but the crab cake sandwich not so good.","Captn Bill's kitchen is a typical beach bar grub spot. We had the charred oysters, crab cake sandwich and lobster bisque. Oysters good, lobster bisque good but the crab cake sandwich not so good. The potatoes and slaw were the side were bland. Drinks were decent priced. Server was great. Place is a must try if your in a beachie kinda mood for grub"
77887,dish_mention_47d0de021639cdea3286,eX9N0rLvcfgrOo34hvDQRQ,ltBBYdNzkeKdCNPDAsxwAA,fries,fries,dish_seed_v2_000003,4.0,positive,positive,2.1920,0.8849,high,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern]",[good:positive_lexicon],[],The sweet potato fries are good too,but I've heard it's delish. The sweet potato fries are good too but share them!,The sweet potato fries are good too but share them!,The kale salad is life changing. This restaurant is a bit pricey but it's such a good go-to spot that I can't really complain. I haven't had the brunch but I've heard it's delish. The sweet potato fries are good too but share them!
83637,dish_mention_b49811bdf87c7f832e16,aSALQMv7xqw_jTLaRbQDwQ,GVqXNt0HKJLpciN9ePxnWw,fried chicken,fried chicken,dish_seed_v2_000019,4.0,positive,positive,1.5670,0.7726,high,target_context_primary,[contrast_marker_nearby],"[best:positive_lexicon, crispy:positive_lexicon, tender:positive_lexicon]",[],This is some of the best fried chicken in the city.,"This is some of the best fried chicken in the city. Really tender pieces. I'm normally not a skin person, but the crispy flavor",This is some of the best fried chicken in the city.,"This is some of the best fried chicken in the city. Really tender pieces. I'm normally not a skin person, but the crispy flavor is just right. Definitely worth the wait and the trek over."
30094,dish_mention_7c243e54a0a88eb6ee64,ExFSsV94IMaxZszczNC7-A,EZuIzN2oAdZt-dkbdwH0pw,tacos,tacos,dish_seed_v2_000005,4.0,positive,negative,-1.5080,0.7640,high,target_context_primary,[],[],"[favorite:negated_positive, greasy:negative_lexicon]",") We shared the salmon tacos (YUM), another experience my husband had shredded pork tacos (I think) and I had the Rueben (not my favorite as the bread was buttered and slightly greasy.",ther experience my husband had shredded pork tacos (I think) and I had the Rueben (not my favorite as the bread was buttered and slightly gr,") We shared the salmon tacos (YUM), another experience my husband had shredded pork tacos (I think) and I had the Rueben (not my favorite as the bread was buttered and slightly greasy.","Visiting family in Boise, Idaho for two weeks and we ate here twice. I was shocked at the low prices for the seasonal ($8.00) and regular beer samplers ($5.00.) I had both during my 2 visits as I love tasting different beers. My husband enjoyed the IPA's. The food was excellent. We had the calamari for appetizer (cooked perfectly and light.) We..."
19713,dish_mention_2d09cc960ee5a3c0e507,8dOiZdNYV0UsxkDUwmM7Hw,e86IBzGCsrnhJbD_wELj7w,oysters,oysters,dish_seed_v2_000011,5.0,positive,positive,2.7630,0.9160,high,target_cont

,mention_id,review_id,business_id,dish_mention_text,canonical_dish_name_v2,dish_id_v2,rating_value,sentiment_label,mention_sentiment_label_final,mention_sentiment_score_final,mention_sentiment_confidence_final,sentiment_reliability_tier,sentiment_reason_final,sentiment_flags_final,positive_terms_v11,negative_terms_v11,target_clause_context_v11,near_mention_context_v11,sentence_context,review_text
48536,dish_mention_edf11ac17207bdd44436,cu8_9kdlnT3fQ-ffM5xy1w,w0lcP2ngFaUpBGi5yzlYDw,chicken,chicken,dish_seed_v2_000066,3.0,neutral,neutral,0.000,0.1660,low,review_prior_fallback,"[ambiguous_mixed_resolved_as_neutral, contrast_marker_nearby, low_ner_confidence, mixed_target_evidence, used_review_prior_fallback, weak_target_sentiment_signal]",[delicious:positive_lexicon],[overpriced:negative_lexicon],"I had the sampler with curry chicken, ox tail, & jerk chicken.","sampler with curry chicken, ox tail, & jerk chicken. The rice & peas is ok. The mac & cheese was delicious but at $6 was overpriced. The high","I had the sampler with curry chicken, ox tail, & jerk chicken.",Checked this place out based on Yelp reviews. The location is strange. It is hidden in the back of a plaza behind the UPS store and across from Walgreens. The decor inside is very nice. Beautiful decorations and a dope upscale vibe. There is several TVs which had sports on. I dig it. The food selections are great. The food is tasty. There was a...
49709,dish_mention_23683051af7aa358b4ca,vNkNgVD2poNEDrdBCUsDBw,Wk21f0DAM7uj3DaJ_rMI-Q,eggs,egg,dish_seed_v2_000118,3.0,neutral,neutral,0.000,0.1713,low,review_prior_fallback,"[ambiguous_mixed_resolved_as_neutral, contrast_marker_nearby, low_ner_confidence, mixed_target_evidence, used_review_prior_fallback, weak_target_sentiment_signal]",[good:positive_lexicon],[hard:negative_lexicon],I had eggs potatoes and blueberry pancakes.,dly and helpful. The food was just ok. I had eggs potatoes and blueberry pancakes. The eggs were good but the potatoes were kind of hard an,I had eggs potatoes and blueberry pancakes.,Very clean restaurant and the server and staff were super friendly and helpful. The food was just ok. I had eggs potatoes and blueberry pancakes. The eggs were good but the potatoes were kind of hard and just warm. Pancakes were alright. I know others really enjoyed the food so I would give it another shot.
74953,dish_mention_ab788babf286a51d6e18,ABfeHBgsb5jWLQ4Xjw51AQ,Jy1Cu7KvaVlYCyze8-3krQ,boxcar burger,boxcar burger,dish_seed_v2_002398,3.0,neutral,neutral,0.000,0.1719,low,review_prior_fallback,"[ambiguous_mixed_resolved_as_neutral, contrast_marker_nearby, low_ner_confidence, mixed_target_evidence, used_review_prior_fallback, weak_target_sentiment_signal]",[flavorful:positive_lexicon],[tough:negative_lexicon],I had boxcar burger which was on special.,g the patio seemed spread pretty thin. I had boxcar burger which was on special. It was flavorful but a little tough. The ranch potatoes were really,I had boxcar burger which was on special.,Stopped here for lunch after a group bike ride. The outdoor patio is very nice with a mix of sun and shade. The one waitress covering the patio seemed spread pretty thin. I had boxcar burger which was on special. It was flavorful but a little tough. The ranch potatoes were really good.
60066,dish_mention_0aadc737b5f1968b6ff2,N2GCUGBOmVB_ubvRroTDKw,UCMSWPqzXjd7QHq7v8PJjQ,done,done,dish_seed_v2_000718,2.0,negative,neutral,-0.131,0.1721,low,review_prior_fallback,"[ambiguous_mixed_resolved_as_neutral, contrast_marker_nearby, low_ner_confidence, mixed_target_evidence, used_review_prior_fallback, weak_target_sentiment_signal]",[good:positive_lexicon],[soggy:negative_lexicon],"The eggs were too well done,","ato Hash with 2 eggs. The eggs were too well done, but the hash was good. I also had a biscuit, which was soggy. My husband had French Toas","The eggs were too well done, but the hash was good.","I ordered the Sweet Potato Hash with 2 eggs. The eggs were too well done, but the hash was good.

,mention_id,review_id,business_id,dish_mention_text,canonical_dish_name_v2,dish_id_v2,rating_value,sentiment_label,mention_sentiment_label_final,mention_sentiment_score_final,mention_sentiment_confidence_final,sentiment_reliability_tier,sentiment_reason_final,sentiment_flags_final,positive_terms_v11,negative_terms_v11,target_clause_context_v11,near_mention_context_v11,sentence_context,review_text
49341,dish_mention_4510453a886f1f529467,5eT2E-ISenh_ovrE3dgLLw,TUhRukcZb3tG0cB_X68FCQ,salmon skin pressed sushi triangle,salmon skin pressed sushi triangle,dish_seed_v2_007838,3.0,neutral,positive,3.0875,0.95,high,target_context_primary,[direct_positive_pattern],"[delicious:positive_lexicon, good:positive_lexicon, hot:positive_lexicon, positive_phrase:positive_phrase]",[],The salmon skin pressed sushi triangle was delicious.,"a server will deliver it to your table. The salmon skin pressed sushi triangle was delicious. The wings in the hot foods section was also pretty good. I will agree, tha",The salmon skin pressed sushi triangle was delicious.,"Aside from being sparkling new, what really distinguishes Noboru from the more well known sushi buffet in the suburban Philly area (Minado) is the selection of Korean dishes. You can get a made to order stew (i.e. Korean seafood tofu). The person behind the counter will give you a number and when it's ready, a server will deliver it to your tab..."
69605,dish_mention_3160292d6d9d48368a2f,IxpbQN2W98AD4z0HqZ8ULA,kKPJLiHIr9Gd9sYs3GhxDQ,burger,burger,dish_seed_v2_000002,3.0,neutral,positive,2.9625,0.95,high,target_context_primary,[direct_positive_pattern],"[delicious:positive_lexicon, positive_phrase:positive_phrase, recommend:positive_lexicon]",[],We had the burger and truffle fries which were delicious!,so I would recommend to go early. We had the burger and truffle fries which were delicious! My friend had the shrimp salad with half avacado,We had the burger and truffle fries which were delicious!,Cute restaurant with new age fusion food. Good selection of wine and beer however they do not have liquor. The restaurant does not take reservations and is small and seems to fill up fast so I would recommend to go early. We had the burger and truffle fries which were delicious! My friend had the shrimp salad with half avacado but the avacado w...
2746,dish_mention_63aa2364ac0267a442fe,6hpp-DZ-ayT8rZmLiV8s9g,Dv6RfXLYe1atjgz3Xf4GGw,burger,burger,dish_seed_v2_000002,3.0,neutral,positive,5.0000,0.95,high,target_context_primary,[direct_positive_pattern],"[fresh:positive_lexicon, good:positive_lexicon, loved:positive_lexicon, positive_phrase:positive_phrase]",[],"I've also had the ziggy burger which I thought was pretty good for a veggie burger (loved the fresh tomato, lettuce and onion)","d with faux-chicken. I've also had the ziggy burger which I thought was pretty good for a veggie burger (loved the fresh tomato, lettuce and","I've also had the ziggy burger which I thought was pretty good for a veggie burger (loved the fresh tomato, lettuce and onion) but especially enjoyed the sweet potato fries I had as a side.",This is a good place to get a take out lunch if you are in the mood for something healthy. I would not recommend eating there because seating is limited and there are crowds of people waiting to order/pick up their take out meals. Rittenhouse Square is just a few steps away and that's where I go to eat my lunch. The lady sitting next to me on t...
2747,dish_mention_fa43a2859f19e354e89d,6hpp-DZ-ayT8rZmLiV8s9g,Dv6RfXLYe1atjgz3Xf4GGw,burger,burger,dish_seed_v2_000002,3.0,neutral,positive,5.0000,0.95,high,target_context_primary,"[contrast_marker_nearby, direct_positive_pattern]","[enjoyed:positive_lexicon, fresh:positive_lexicon, good:positive_lexicon, loved:positive_lexicon, positive_phrase:positive_phrase]",[],"I've also had the ziggy burger which I thought was pretty good for a veggie burger (loved the fresh tomato, lettuce and onion)","which I thought was pretty good for a veggie burger (love

In [53]:
# ============================================================
# 30. Preparar dataset final compacto
# ============================================================

final_output_cols = [
    # IDs
    "mention_id",
    "corpus_document_id",
    "review_id",
    "business_id",

    # negocio / fuente
    "business_name",
    "city",
    "state",
    "split",
    "language",
    "source_system_code",
    "source_dataset",

    # plato
    "dish_mention_text",
    "dish_mention_normalized",
    "canonical_dish_name_v2",
    "dish_id_v2",
    "normalization_status_v2",
    "normalization_method_v2",

    # posición y NER
    "start_char",
    "end_char",
    "start_token",
    "end_token",
    "token_count",
    "confidence_mean",
    "confidence_min",
    "confidence_max",

    # review
    "rating_value",
    "sentiment_label",
    "review_text",

    # contexto
    "sentence_context",
    "window_context",
    "target_clause_context_v11",
    "near_mention_context_v11",

    # sentimiento final
    "mention_sentiment_label_final",
    "mention_sentiment_score_final",
    "mention_sentiment_confidence_final",
    "sentiment_reliability_tier",
    "sentiment_reason_final",
    "sentiment_flags_final",
    "sentiment_method_final",

    # señales
    "positive_terms_v11",
    "negative_terms_v11",
    "target_signal_strength_v11",
    "rating_prior_score_v11",
    "review_prior_score_v11",
    "direct_positive_count_v11",
    "direct_negative_count_v11",

    # futuro entrenamiento
    "is_sentiment_training_candidate",
    "human_review_status",
]

# Algunas columnas pueden no estar en todos los entornos; las creamos vacías.
for col in final_output_cols:
    if col not in mentions_final_df.columns:
        mentions_final_df[col] = None

mentions_with_sentiment_final_df = mentions_final_df[final_output_cols].copy()

print("Dataset final compacto:", mentions_with_sentiment_final_df.shape)

display(mentions_with_sentiment_final_df.head(5))

Dataset final compacto: (94932, 48)


,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,canonical_dish_name_v2,dish_id_v2,normalization_status_v2,normalization_method_v2,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,rating_value,sentiment_label,review_text,sentence_context,window_context,target_clause_context_v11,near_mention_context_v11,mention_sentiment_label_final,mention_sentiment_score_final,mention_sentiment_confidence_final,sentiment_reliability_tier,sentiment_reason_final,sentiment_flags_final,sentiment_method_final,positive_terms_v11,negative_terms_v11,target_signal_strength_v11,rating_prior_score_v11,review_prior_score_v11,direct_positive_count_v11,direct_negative_count_v11,is_sentiment_training_candidate,human_review_status
0,dish_mention_c6a233f28ab0ec50a4bf,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,en,yelp_open_dataset,yelp_open_dataset,tamale,tamale,tamale,dish_seed_v2_000060,normalized_rule_based_v2,rule_based_v2_overrides,88,94,18,19,1,0.997417,0.997417,0.997417,3.0,neutral,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with no expectations. Next to the Clarion Hotel.","Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served a","Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.","rtment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has",positive,1.1500,0.7066,medium,target_context_primary,[],hybrid_context_lexicon_v1_1,"[fresh:positive_lexicon, good:positive_lexicon]",[],1.8,0.00,0.00,0,0,False,not_reviewed
1,dish_mention_370e33f760028e4e87a3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,en,yelp_open_dataset,yelp_open_dataset,lamb curry,lamb curry,lamb curry,dish_seed_v2_000205,normalized_rule_based_v2,rule_based_v2_overrides,54,64,12,14,2,0.996373,0.993416,0.999331,5.0,positive,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",Our favorite is the lamb curry and korma.,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...g",Our favorite is the lamb curry and korma.,"y, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we",positive,1.2130,0.7128,medium,target_context_primary,[],hybrid_context_lexicon_v1_1,"[delicious:positive_lexicon, favorite:positive_lexicon]",[],1.8,0.75,0.35,0,0,False,not_reviewed
2,dish_mention_2c51d464763786d9fef2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,en,yelp_open_dataset,yelp_open_dataset,korma,korma,korma,dish_seed_v2_000074,normalized_rule_based_v2,rule_based_v2_overrides,69,74,15,16,1,0.997466,0.997466,0.997466,5.0,positive,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something ne

In [54]:
# ============================================================
# 31. Guardar artefactos finales
# ============================================================

mentions_sentiment_path = MENTIONS_DIR / "dish_mentions_with_sentiment_hybrid_v1.jsonl"
summary_path = ARTIFACTS_DIR / "dish_mention_sentiment_summary_hybrid_v1.json"

sample_random_path = SAMPLES_DIR / "dish_mention_sentiment_sample_random_v1.jsonl"
sample_high_conf_path = SAMPLES_DIR / "dish_mention_sentiment_sample_high_confidence_v1.jsonl"
sample_low_conf_path = SAMPLES_DIR / "dish_mention_sentiment_sample_low_confidence_v1.jsonl"
sample_disagreement_path = SAMPLES_DIR / "dish_mention_sentiment_sample_review_disagreement_v1.jsonl"
sample_mixed_path = SAMPLES_DIR / "dish_mention_sentiment_sample_mixed_evidence_v1.jsonl"
training_candidates_path = SAMPLES_DIR / "dish_mention_sentiment_training_candidates_v1.jsonl"

save_jsonl(mentions_with_sentiment_final_df, mentions_sentiment_path)
save_jsonl(sample_random_df, sample_random_path)
save_jsonl(sample_high_conf_df, sample_high_conf_path)
save_jsonl(sample_low_conf_df, sample_low_conf_path)
save_jsonl(sample_disagreement_df, sample_disagreement_path)
save_jsonl(sample_mixed_df, sample_mixed_path)
save_jsonl(sentiment_training_candidates_df, training_candidates_path)

sentiment_summary_output = {
    "notebook": "09_dish_mention_sentiment_hybrid_v1",
    "version": "hybrid_context_lexicon_v1_1",
    "source_mentions_path": str(MENTIONS_NORMALIZED_PATH),
    "input": qa_initial,
    "final_sentiment_qa": final_sentiment_qa,
    "outputs": {
        "mentions_with_sentiment": str(mentions_sentiment_path),
        "summary": str(summary_path),
        "sample_random": str(sample_random_path),
        "sample_high_confidence": str(sample_high_conf_path),
        "sample_low_confidence": str(sample_low_conf_path),
        "sample_review_disagreement": str(sample_disagreement_path),
        "sample_mixed_evidence": str(sample_mixed_path),
        "training_candidates": str(training_candidates_path),
    },
    "notes": [
        "The final label uses the refined v1.1 hybrid sentiment assignment.",
        "This is a weak-supervised aspect-level sentiment signal, not a manually validated gold label.",
        "The reliability tier should be used downstream to weight signals in aggregation and ranking.",
        "High-confidence candidates can be reused later to train an aspect-based sentiment model."
    ]
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(sentiment_summary_output, f, indent=2, ensure_ascii=False)

print("Artefactos guardados:")
print("mentions_sentiment_path:", mentions_sentiment_path)
print("summary_path:", summary_path)
print("training_candidates_path:", training_candidates_path)

print("\nResumen:")
print(json.dumps(sentiment_summary_output, indent=2, ensure_ascii=False)[:6000])

Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sentiment_hybrid/mentions/dish_mentions_with_sentiment_hybrid_v1.jsonl (94,932 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sentiment_hybrid/samples/dish_mention_sentiment_sample_random_v1.jsonl (300 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sentiment_hybrid/samples/dish_mention_sentiment_sample_high_confidence_v1.jsonl (300 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sentiment_hybrid/samples/dish_mention_sentiment_sample_low_confidence_v1.jsonl (400 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sentiment_hybrid/samples/dish_mention_sentiment_sample_review_disagreement_v1.jsonl (400 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sentiment_hybrid/samples/dish_mention_sentiment_sample_mixed_evidence_v1.jsonl (400 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_mention_sen

In [55]:
# ============================================================
# 32. Comprimir outputs del módulo 09
# ============================================================

import shutil

zip_base = Path("/kaggle/working") / "dish_mention_sentiment_hybrid_v1_outputs"

if zip_base.with_suffix(".zip").exists():
    zip_base.with_suffix(".zip").unlink()

shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

print("ZIP generado:")
print(str(zip_base) + ".zip")

ZIP generado:
/kaggle/working/dish_mention_sentiment_hybrid_v1_outputs.zip


## Cierre del Notebook 09

En este notebook se ha construido la primera versión del módulo de sentimiento por mención de plato.

El flujo realizado ha sido:

1. Carga de menciones normalizadas v2.
2. Extracción de contexto local.
3. Definición de léxicos positivos y negativos.
4. Asignación híbrida inicial v1.
5. Refinamiento aspect-based v1.1.
6. Creación de etiqueta final de sentimiento por mención.
7. Asignación de nivel de fiabilidad.
8. Generación de muestras de revisión.
9. Preparación de candidatos para entrenamiento futuro.

El archivo principal generado es:

`dish_mentions_with_sentiment_hybrid_v1.jsonl`

Este archivo será la entrada del siguiente módulo:

`10_dish_signal_aggregation.ipynb`

donde se agregarán señales por plato, negocio y, más adelante, barrio.